# HDB Rentals 0: shared code

The code that notebooks 1-3 share. Each of them starts with `%run 0_shared_code.ipynb`, so you never need to run this
notebook yourself.

Every cell below that starts with `%%module <name>` is one module, imported elsewhere as `hdb_rentals.<name>`, e.g.
`from hdb_rentals import gnn`. A module's code runs the first time it is imported, not when this notebook runs, so
`%run` only registers the code: no data is loaded and nothing is trained here. Modules keep their own namespaces
because several define functions with the same name (`main`, `train`, `prepare`).

Experiment modules have a `main(argv)` that takes the same options the old command-line scripts did, e.g.
`gnn_premiums_blend.main(["--level", "blend", "--same-level", "--seed", "1"])`. Notebook 2 (Part 2.4) and notebook 3
call them behind `RUN_...`/`RETRAIN` flags that are off by default, so the frozen results in `outputs/` are read
rather than recomputed.

| Section | Modules | Used by |
|---|---|---|
| 1. Data and validation | `config`, `data`, `validation`, `tracking`, `macro` | 1, and the modules below |
| 2. Baseline and market level | `baselines`, `market_level`, `rent_adjustment` | 1, 3 (Appendix D), GNNs |
| 3. Graph neural networks | `block_evaluation`, `gnn`, `explain`, `graph_data`, `experiment_weighted_gnns`, `compare_gnn_lightgbm`, `gnn_premiums_blend`, `gnn_neighbour_premiums`, `gnn_graph_tests`, `explain_table2_gnn` | 2 (Part 2.4), 3 (Parts 3.2, 3.3) |
| 4. Market-level evaluation for the report | `report_level_eval` | 3 (Appendix D) |
| 5. Geographic transfer | `transfer_models`, `transfer` | 2 (end of Part 2.4) |

Import order matters on some machines: torch must be imported before LightGBM, which is why the next cell does it.


In [ ]:
import torch  # first: torch must load its OpenMP runtime before LightGBM does

import importlib.abc
import importlib.util
import linecache
import sys
from pathlib import Path

from IPython.core.magic import register_cell_magic

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
MODULE_SOURCES = {}


class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    """Imports `hdb_rentals.<name>` from the `%%module <name>` cell below, on first use."""

    def find_spec(self, name, path=None, target=None):
        if name == "hdb_rentals":
            return importlib.util.spec_from_loader(name, self, is_package=True)
        if name.startswith("hdb_rentals.") and name.split(".", 1)[1] in MODULE_SOURCES:
            return importlib.util.spec_from_loader(name, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        if module.__name__ == "hdb_rentals":
            return
        short = module.__name__.split(".", 1)[1]
        source = MODULE_SOURCES[short]
        filename = f"<0_shared_code.ipynb: {short}>"
        linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)  # readable tracebacks
        module.ROOT = ROOT
        exec(compile(source, filename, "exec"), module.__dict__)


sys.meta_path[:] = [f for f in sys.meta_path if type(f).__name__ != "NotebookModules"]
sys.meta_path.insert(0, NotebookModules())


@register_cell_magic
def module(name, cell):
    """Store the cell as module `hdb_rentals.<name>`. It runs when first imported, so nothing below loads data yet."""
    name = name.strip()
    MODULE_SOURCES[name] = cell
    sys.modules.pop(f"hdb_rentals.{name}", None)  # re-running a cell redefines its module


del module

## 1. Data and validation

Paths, loading and the time-ordered splits.

In [ ]:
%%module config
"""Project paths and shared constants."""

from pathlib import Path

PROJECT_ROOT = ROOT  # set by 0_shared_code.ipynb

DATA_DIR = PROJECT_ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"

SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

AUX_CSV = {
    "hdb_block": AUX_DIR / "sg-hdb-block.csv",
    "mrt": AUX_DIR / "sg-mrt-stations.csv",
    "schools": AUX_DIR / "sg-schools.csv",
    "malls": AUX_DIR / "sg-shopping-malls.csv",
    "coe": AUX_DIR / "sg-coe-prices.csv",
    "stocks": AUX_DIR / "sg-stock-prices.csv",
}

TARGET = "MONTHLY_RENT"

# Columns that carry no signal in the data as shipped:
#   FEE      is 0.0 for every row of train and test
#   FURNISHED is "yes" for every row of train and test
CONSTANT_COLUMNS = ["FEE", "FURNISHED"]

# The natural join key onto sg-hdb-block.csv. Matches 100% of train and test
# rows once all three parts are lowercased and stripped.
BLOCK_KEY = ["TOWN", "BLOCK", "STREET"]

# The test set begins where the training set ends, so validation must be
# ordered in time rather than shuffled. See validation.py.
TRAIN_PERIOD = ("2021-01", "2025-03")
TEST_PERIOD = ("2025-03", "2026-07")

In [ ]:
%%module data
"""Loading and normalisation for the HDB rental data.

The raw CSVs carry three quirks that will silently cost accuracy if they are
not handled at load time:

1. ``FLAT_TYPE`` is encoded two ways in train -- "4-room" (40,688 rows) and
   "4 room" (13,391 rows) are the same category. Test uses only the hyphenated
   form. Left alone, a model treats them as unrelated levels and the "4 room"
   rows teach it nothing usable at prediction time.
2. ``STREET`` is mixed case in both files ("Whampoa Drive" vs "whampoa drive"),
   which breaks the join onto the block geodata.
3. ``FEE`` and ``FURNISHED`` are constant, so they are dropped.

``load_train`` and ``load_test`` apply all of this; use them rather than
reading the CSVs directly.
"""

from __future__ import annotations

import pandas as pd

from . import config

_TEXT_COLUMNS = ["TOWN", "BLOCK", "STREET", "FLAT_TYPE", "FLAT_MODEL", "FURNISHED"]

# "4 room" -> "4-room". Applied after lowercasing.
_FLAT_TYPE_CANONICAL = {f"{n} room": f"{n}-room" for n in range(1, 6)}


def _normalise_text(df: pd.DataFrame) -> pd.DataFrame:
    for col in _TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip().str.lower()
    if "FLAT_TYPE" in df.columns:
        df["FLAT_TYPE"] = df["FLAT_TYPE"].replace(_FLAT_TYPE_CANONICAL)
    return df


def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Expand RENT_APPROVAL_DATE ("2021-01") into usable time columns.

    ``MONTH_INDEX`` counts months from 2021-01 and is the column the
    time-ordered splits in validation.py sort on.
    """
    period = pd.PeriodIndex(df["RENT_APPROVAL_DATE"], freq="M")
    origin = pd.Period("2021-01", freq="M")
    df["APPROVAL_PERIOD"] = period
    df["YEAR"] = period.year
    df["MONTH"] = period.month
    df["MONTH_INDEX"] = (
        period.year * 12 + period.month - (origin.year * 12 + origin.month)
    )
    # Age of the flat's lease at the time of the rental approval, in years.
    df["FLAT_AGE"] = df["YEAR"] - df["LEASE_COMMENCE_DATE"]
    return df


def _load_split(path, *, drop_constant: bool) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = _normalise_text(df)
    df = _add_time_features(df)
    if drop_constant:
        df = df.drop(columns=[c for c in config.CONSTANT_COLUMNS if c in df.columns])
    return df


def load_train(*, drop_constant: bool = True) -> pd.DataFrame:
    """Training rows, normalised, sorted oldest first."""
    df = _load_split(config.TRAIN_CSV, drop_constant=drop_constant)
    return df.sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)


def load_test(*, drop_constant: bool = True) -> pd.DataFrame:
    """Test rows in their original file order.

    Order is preserved because submissions are matched by row position.
    """
    return _load_split(config.TEST_CSV, drop_constant=drop_constant)


def load_auxiliary(name: str) -> pd.DataFrame:
    """Load one auxiliary table by key, e.g. ``load_auxiliary("mrt")``."""
    if name not in config.AUX_CSV:
        raise KeyError(
            f"unknown auxiliary table {name!r}; "
            f"expected one of {sorted(config.AUX_CSV)}"
        )
    df = pd.read_csv(config.AUX_CSV[name])
    return _normalise_text(df)


def attach_block_geodata(df: pd.DataFrame) -> pd.DataFrame:
    """Left-join latitude/longitude and block attributes onto rental rows.

    The join key is unique in sg-hdb-block.csv and covers 100% of both splits,
    so this neither drops nor duplicates rows -- both are asserted.
    """
    blocks = load_auxiliary("hdb_block")
    n_before = len(df)
    merged = df.merge(blocks, on=config.BLOCK_KEY, how="left", validate="many_to_one")
    assert len(merged) == n_before, "block join changed the row count"
    missing = merged["LATITUDE"].isna().sum()
    assert missing == 0, f"{missing} rows failed to match a block"
    return merged

In [ ]:
%%module validation
"""Time-ordered validation.

The test set starts where training ends -- train runs 2021-01 to 2025-03,
test runs 2025-03 to 2026-07 -- so every test prediction is an extrapolation
forward in time. These splitters score that same forward horizon, which makes
the validation number comparable to the one the leaderboard will report.

They are not here to prevent leakage. Letting a model see labelled rows from
the month it is scored on was measured at roughly 1.6 RMSE against a 464
baseline -- a third of a percent, and not worth engineering around. Rent rose
steeply from 2021 to mid-2023 and has been flat since, so knowing the month
carries little information at the recent end where both validation and test
live.

The exposure these splits genuinely cannot measure is extrapolation. A tree
model cannot predict above the largest MONTH_INDEX it trained on; it returns
the value of its last leaf for month 51 and for month 100 alike. Every test
row is past that boundary, so a plain time feature freezes all 50,000
predictions at the 2025-03 level for sixteen months. Validating on held-out
history will not reveal this, because history stops where the extrapolation
begins.
"""

from __future__ import annotations

from collections.abc import Iterator, Sequence

import numpy as np
import pandas as pd


def holdout_split(
    df: pd.DataFrame, *, n_valid_months: int = 4, month_col: str = "MONTH_INDEX"
) -> tuple[np.ndarray, np.ndarray]:
    """Train on everything before the last ``n_valid_months``, validate on them.

    Four months is the default because it is the horizon the test set opens
    with; widen it to stress-test further-out predictions.
    """
    months = df[month_col].to_numpy()
    cutoff = np.sort(np.unique(months))[-n_valid_months]
    return np.flatnonzero(months < cutoff), np.flatnonzero(months >= cutoff)


def expanding_window_splits(
    df: pd.DataFrame,
    *,
    n_splits: int = 5,
    valid_months: int = 3,
    month_col: str = "MONTH_INDEX",
) -> Iterator[tuple[np.ndarray, np.ndarray]]:
    """Yield ``n_splits`` (train_idx, valid_idx) pairs that walk forward in time.

    Each fold trains on all months up to a cutoff and validates on the next
    ``valid_months``. The training window grows with each fold, mirroring how
    the final model is fit on all history before predicting the future.
    """
    months = df[month_col].to_numpy()
    unique = np.sort(np.unique(months))
    last = len(unique)
    for k in range(n_splits, 0, -1):
        v_end = last - (k - 1) * valid_months
        v_start = v_end - valid_months
        if v_start <= 0:
            raise ValueError(
                f"not enough months ({last}) for {n_splits} folds of {valid_months}"
            )
        v_months = set(unique[v_start:v_end])
        cutoff = unique[v_start]
        yield (
            np.flatnonzero(months < cutoff),
            np.flatnonzero(np.isin(months, list(v_months))),
        )


def horizon_matched_split(
    df: pd.DataFrame, *, horizon_months: int = 16, month_col: str = "MONTH_INDEX"
) -> tuple[np.ndarray, np.ndarray]:
    """One fold whose forecast distance matches the real test set.

    The expanding-window folds ask the model to predict 1-3 months ahead; the
    test set asks for 1-16. That mismatch does not change how configurations
    rank -- measured, full features beat fewer features at both distances --
    so the short folds remain the right tool for iteration.

    What the short folds hide is bias. Trained through month 33, the model
    under-predicts by $0 at three months out and by $75 at sixteen, growing
    monotonically, because it cannot extrapolate past its last training month.
    RMSE actually *falls* over the same range (557 -> 478) because rent is less
    variable in the later months, so watching RMSE alone suggests the long
    horizon is fine when it is not.

    Run this occasionally as a reality check rather than on every iteration,
    and read ``horizon_report`` alongside it.
    """
    months = df[month_col].to_numpy()
    cutoff = months.max() - horizon_months + 1
    if cutoff <= months.min():
        raise ValueError(f"horizon of {horizon_months} months leaves no training data")
    return np.flatnonzero(months < cutoff), np.flatnonzero(months >= cutoff)


def horizon_report(
    df: pd.DataFrame,
    y_true,
    y_pred,
    valid_idx: np.ndarray,
    *,
    buckets: Sequence[tuple[int, int]] = ((1, 3), (4, 6), (7, 9), (10, 12), (13, 16)),
    month_col: str = "MONTH_INDEX",
) -> pd.DataFrame:
    """Break error down by how far ahead each row was predicted.

    Returns RMSE, RMSE normalised by the target's own spread, and mean signed
    bias per bucket. The bias column is the point of this function: a model
    that cannot extrapolate drifts one direction as the horizon grows, and a
    single aggregate RMSE will not show it.
    """
    months = df[month_col].to_numpy()
    last_trained = months[np.setdiff1d(np.arange(len(df)), valid_idx)].max()
    ahead = months[valid_idx] - last_trained
    t = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)

    rows = []
    for lo, hi in buckets:
        sel = (ahead >= lo) & (ahead <= hi)
        if not sel.any():
            continue
        rows.append(
            {
                "horizon": f"+{lo}..+{hi}",
                "n": int(sel.sum()),
                "target_sd": round(float(t[sel].std()), 1),
                "rmse": round(rmse(t[sel], p[sel]), 1),
                "rmse_over_sd": round(float(rmse(t[sel], p[sel]) / t[sel].std()), 3),
                "bias": round(float((p[sel] - t[sel]).mean()), 1),
            }
        )
    return pd.DataFrame(rows)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

In [ ]:
%%module tracking
"""Append-only experiment log, shared through git.

Every run appends one JSON object to ``outputs/experiments.jsonl``.
The log is the team's shared record of what has been tried, so the rule is:
if a run produced a score you would quote to a teammate, log it.

Why one line of JSON per run rather than a Markdown table: two people editing
the same table conflict on nearly every merge. Appended lines are handled by
the ``merge=union`` rule in ``.gitattributes``, which keeps both sides'
entries instead of asking anyone to resolve them.

Each entry records the git commit and whether the tree was dirty. A score
without the code that produced it cannot be reproduced or trusted, and on a
team it cannot even be attributed.
"""

from __future__ import annotations

import json
import subprocess
from collections.abc import Sequence
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from . import config

LOG_PATH = config.PROJECT_ROOT / "outputs" / "experiments.jsonl"


def _git(*args: str) -> str:
    try:
        return subprocess.run(
            ["git", *args],
            cwd=config.PROJECT_ROOT,
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return ""


def _git_state() -> dict[str, Any]:
    """Commit, branch and cleanliness, so a result can be traced to its code.

    The log file itself is excluded from the dirty check. ``dirty`` is meant to
    answer "could I check this code out again and re-run it", and appending a
    line to the log is not a code change. Without the exclusion the first entry
    of a session records clean and every entry after it records dirty, because
    the log dirtied the tree by writing to itself.
    """
    return {
        "commit": _git("rev-parse", "--short", "HEAD"),
        "branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
        "dirty": bool(_git("status", "--porcelain", f":(exclude)outputs/experiments.jsonl")),
        "author": _git("config", "user.name") or _git("config", "user.email"),
    }


def log_experiment(
    *,
    name: str,
    score: float,
    validation: str,
    features: Sequence[str],
    model: str,
    params: dict[str, Any] | None = None,
    fold_scores: Sequence[float] | None = None,
    metric: str = "rmse",
    notes: str = "",
    extra: dict[str, Any] | None = None,
    path: Path | None = None,
) -> dict[str, Any]:
    """Append one run to the experiment log and return the entry.

    Args:
        name: Short handle for the run, e.g. ``"lgbm-mrt-distance"``.
        score: The headline number, lower-is-better for RMSE.
        validation: Which split produced it. Scores from different schemes
            are not comparable, so this is required rather than optional --
            see the fold-to-fold spread noted in validation.py.
        features: Columns fed to the model.
        model: Estimator name, e.g. ``"LGBMRegressor"``.
        params: Hyperparameters worth recording.
        fold_scores: Per-fold numbers behind ``score``. Keep these: comparing
            two runs fold by fold cancels fold difficulty and detects far
            smaller differences than comparing the means.
        notes: What you were testing and what you concluded. The field that
            makes the log worth reading in a month.
    """
    entry = {
        "timestamp": datetime.now(UTC).isoformat(timespec="seconds"),
        "name": name,
        "metric": metric,
        "score": round(float(score), 4),
        "fold_scores": [round(float(s), 4) for s in fold_scores] if fold_scores else None,
        "validation": validation,
        "model": model,
        "n_features": len(features),
        "features": sorted(features),
        "params": params or {},
        "notes": notes,
        **_git_state(),
        **(extra or {}),
    }
    target = path or LOG_PATH
    with target.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(entry, sort_keys=False) + "\n")
    return entry


def read_log(path: Path | None = None) -> list[dict[str, Any]]:
    """Read every logged run. Bad lines are skipped rather than raising, so a
    half-written entry from an interrupted run cannot block the whole team."""
    target = path or LOG_PATH
    if not target.exists():
        return []
    rows = []
    for line in target.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows

In [ ]:
%%module macro
"""Monthly macro covariates from the COE and stock-price tables.

These two auxiliary tables are unlike the other four. They carry no per-flat
key -- they join on the month alone -- and they are the only inputs in the
project with **observed values inside the test window**: both run through
2026-07, while the labelled data stops at 2025-03.

That matters because of the extrapolation cliff documented in validation.py.
A tree fed ``MONTH_INDEX`` returns its last leaf for every month past training,
so all 50,000 test rows are predicted at the 2025-03 level. A macro series can
carry a time signal across that boundary instead -- but only if its test-window
values fall *inside the range the model saw in training*. A series that climbs
past its own training maximum re-creates the same cliff under a different name,
and is worse than useless because it looks like it should help.

So the loader gates on range, not on correlation. Measured over 2025-04..2026-07
against the 2021-01..2025-03 range:

    coe cat b   1.00 inside      corr with mean monthly rent  +0.84
    coe cat e   1.00                                          +0.82
    coe cat c   0.76                                          +0.91
    coe cat a   0.29                                          +0.94
    stocks      0.00-0.25 for most names                      up to +0.95

Note the inversion: the highest correlations are the least usable. Cat A and
almost every stock spend the test window above their training maximum. The gate
keeps what can actually carry signal forward, and ``macro_coverage`` reports the
fraction for everything so the decision stays visible rather than silent.

One caveat the gate cannot check. Both series and rent rose together through
mid-2023 and flattened together after, so a strong fit may be nothing more than
two trends sharing a period. That is what the plateau-only and placebo checks
are for; see DECISIONS.md. Range is a necessary condition, not a sufficient one.
"""

from __future__ import annotations

import pandas as pd

from . import config

_MONTHS = {
    m: i + 1
    for i, m in enumerate(
        ["jan", "feb", "mar", "apr", "may", "jun",
         "jul", "aug", "sep", "oct", "nov", "dec"]
    )
}

# Equal-weighted groups. Names listed after 2021-02 are excluded everywhere, so
# an index never changes composition mid-window -- a new listing appearing in
# 2024 would otherwise show up as a jump that has nothing to do with the market.
_BANKS = ["DBS Group", "OCBC Bank", "UOB"]
_PROPERTY = [
    "Ascendas Reit", "CapitaLand Ascott Trust", "CapitaLand Investment Limited",
    "CapitaLand Mall Trust", "City Developments",
    "Frasers Logistics & Industrial Trust", "Hotel Properties Limited",
    "Keppel REIT", "Mapletree Commercial Trust", "Mapletree Industrial Trust",
    "Mapletree Logistics Trust", "Pacific Century Regional Developments",
    "Singapore Land", "Suntec REIT", "UOL Group", "Yanlord",
]

_ORIGIN = pd.Period("2021-01", freq="M")


def _month_index(period: pd.PeriodIndex) -> pd.Index:
    return (period.year * 12 + period.month) - (_ORIGIN.year * 12 + _ORIGIN.month)


def _coe_panel() -> pd.DataFrame:
    """Mean quota premium per month per category, one column each.

    Two bidding rounds run each month. They are averaged rather than taking the
    later round: both are realised clearing prices within the month, and the
    mean is the less arbitrary summary. Stated here because the choice is not
    obviously right and changing it changes every number downstream.
    """
    coe = pd.read_csv(config.AUX_CSV["coe"])
    premium = coe["QUOTA_PREMIUM"].replace(r"[\$,]", "", regex=True).astype(float)
    period = pd.PeriodIndex(
        coe["YEAR"].astype(str)
        + "-"
        + coe["MONTH"].str.lower().map(_MONTHS).astype("Int64").astype(str).str.zfill(2),
        freq="M",
    )
    panel = (
        pd.DataFrame({"period": period, "cat": coe["CATEGORY"], "premium": premium})
        .pivot_table(index="period", columns="cat", values="premium", aggfunc="mean")
    )
    panel.columns = [f"COE_{c.upper()}" for c in panel.columns]
    return panel


def _stock_panel() -> pd.DataFrame:
    """Equal-weighted indices rebased to 100 at 2021-01.

    Levels are rebased so names with different share prices contribute equally;
    an unrebased mean is dominated by whichever stock happens to trade highest.
    """
    st = pd.read_csv(config.AUX_CSV["stocks"], parse_dates=["DATE"])
    st["period"] = st["DATE"].dt.to_period("M")
    monthly = st.pivot_table(
        index="period", columns="NAME", values="ADJUSTED_CLOSE", aggfunc="mean"
    )
    full = monthly.columns[monthly.iloc[0].notna() & monthly.notna().all()]
    rebased = monthly[full] / monthly[full].iloc[0] * 100.0

    groups = {
        "STOCK_BANKS": [n for n in _BANKS if n in rebased.columns],
        "STOCK_PROPERTY": [n for n in _PROPERTY if n in rebased.columns],
        "STOCK_BROAD": list(rebased.columns),
    }
    return pd.DataFrame({k: rebased[v].mean(axis=1) for k, v in groups.items()})


def macro_coverage() -> pd.DataFrame:
    """Per-series diagnostic: can this series carry signal past the cliff?

    ``frac_inside`` is the share of test-window months whose value falls within
    the training-window min/max. Anything below 1.0 is partly frozen: for those
    months the tree returns its edge leaf, exactly as it does for MONTH_INDEX.
    """
    panel = _raw_panel()
    train_end = pd.Period(config.TRAIN_PERIOD[1], freq="M")
    intrain = panel[panel.index <= train_end]
    intest = panel[panel.index > train_end]
    rows = []
    for col in panel.columns:
        lo, hi = intrain[col].min(), intrain[col].max()
        post = intest[col].dropna()
        rows.append(
            {
                "series": col,
                "train_min": round(float(lo), 1),
                "train_max": round(float(hi), 1),
                "test_min": round(float(post.min()), 1),
                "test_max": round(float(post.max()), 1),
                "frac_inside": round(float(((post >= lo) & (post <= hi)).mean()), 2),
            }
        )
    return pd.DataFrame(rows).sort_values("frac_inside", ascending=False)


def _raw_panel() -> pd.DataFrame:
    panel = _coe_panel().join(_stock_panel(), how="outer").sort_index()
    first = pd.Period(config.TRAIN_PERIOD[0], freq="M")
    last = pd.Period(config.TEST_PERIOD[1], freq="M")
    panel = panel.loc[first:last]
    expected = pd.period_range(first, last, freq="M")
    missing = expected.difference(panel.index)
    assert len(missing) == 0, f"macro panel has gaps: {list(missing)}"
    assert not panel.isna().any().any(), "macro panel has missing values"
    return panel


def load_macro(*, gate: bool = True, min_inside: float = 1.0) -> pd.DataFrame:
    """Monthly macro panel indexed by ``MONTH_INDEX``, 2021-01 to 2026-07.

    Args:
        gate: Drop series whose test-window values leave the training range.
            On by default -- an ungated series silently reintroduces the
            extrapolation cliff it was meant to solve.
        min_inside: Required share of test months inside the training range.
            1.0 means no month may fall outside.
    """
    panel = _raw_panel()
    if gate:
        cov = macro_coverage().set_index("series")["frac_inside"]
        keep = [c for c in panel.columns if cov[c] >= min_inside]
        assert keep, f"range gate at {min_inside} kept no series"
        panel = panel[keep]
    panel.index = _month_index(panel.index)
    panel.index.name = "MONTH_INDEX"
    return panel


def attach_macro(df: pd.DataFrame, *, gate: bool = True, min_inside: float = 1.0):
    """Left-join the macro panel onto rental rows on ``MONTH_INDEX``.

    Returns ``(frame, columns)``. Like ``attach_block_geodata`` this asserts no
    row loss and no unmatched row, because a silent NaN here would show up as a
    quiet accuracy loss rather than an error.
    """
    panel = load_macro(gate=gate, min_inside=min_inside)
    n_before = len(df)
    merged = df.merge(panel, on="MONTH_INDEX", how="left", validate="many_to_one")
    assert len(merged) == n_before, "macro join changed the row count"
    unmatched = merged[list(panel.columns)].isna().any(axis=1).sum()
    assert unmatched == 0, f"{unmatched} rows fell outside the macro panel"
    return merged, list(panel.columns)

## 2. Baseline and market level

The reference LightGBM, and rent split into a market level times a flat premium.

In [ ]:
%%module baselines
"""The reference runs every other experiment is compared against.

Until this module existed, the code that fit these models lived outside the
repo, so log entries recorded a commit that could not reproduce them. This is
that code, rebuilt from the settings in ``experiments.jsonl``; it reproduces
every logged baseline number exactly (holdout 465.84, the five expanding-window
folds, horizon-matched 506.18 and its bias table).

It doubles as the environment check. LightGBM is deterministic at these
settings, so ``lgbm-raw`` on the holdout must return 465.84 -- anything else
means this environment differs from the one that produced the reference
numbers, and every comparison against them is void.
"""

from __future__ import annotations

from functools import lru_cache

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from pandas.api.types import CategoricalDtype
from sklearn.dummy import DummyRegressor

from . import config
from .data import load_train
from .validation import (
    expanding_window_splits,
    holdout_split,
    horizon_matched_split,
    horizon_report,
    rmse,
)

RAW_FEATURES = ["FLAT_AGE", "FLAT_MODEL", "FLAT_TYPE", "FLOOR_AREA_SQM", "MONTH_INDEX", "TOWN"]

# Passed to LightGBM as pandas categories. The logged numbers depend on this:
# it is the one setting the log did not record, and the exact reproduction is
# the evidence that this is what the original runs did.
CATEGORICAL = ["FLAT_MODEL", "FLAT_TYPE", "TOWN"]

LGBM_PARAMS = {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63}

# Validation labels as they appear in the log. Scores only compare within one.
HOLDOUT = "holdout-last-4-months"
EXPANDING = "expanding-window-5x3"
HORIZON = "horizon-matched-16m"


@lru_cache(maxsize=1)
def category_levels() -> dict[str, CategoricalDtype]:
    """The level set for each categorical column, fixed once from training data.

    LightGBM never sees the string "maisonette"; it sees the integer code
    pandas assigned it, and ``astype("category")`` assigns codes by sorted
    order of the levels *present in the frame it is called on*. Test is missing
    ``improved maisonette``, which train has, so a test matrix built from its
    own levels shifts 14 of the 20 FLAT_MODEL codes by one: code 7 means
    ``maisonette`` to the model and ``model a`` to the test frame. Nothing
    raises, nothing is null, and most maisonette and model-a rows predict as
    the wrong flat model.

    Taking the levels from train and reusing them everywhere keeps one mapping.
    It also leaves the logged numbers untouched, because the training frame's
    own levels are what produced them.
    """
    train = load_train()
    return {c: CategoricalDtype(sorted(train[c].dropna().unique())) for c in CATEGORICAL}


def model_matrix(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Select ``features`` and cast the text columns to categories.

    Built once over the whole frame and then sliced, so every fold shares the
    same category codes -- and every *frame* shares them too, train and test
    alike, because the levels come from ``category_levels`` rather than from
    whatever happens to be in ``df``.

    A level absent from that fixed set would be cast to null and predicted as
    missing, so it raises instead. The trade-off is deliberate: a loud failure
    the first time an unseen category appears, rather than a quiet one at
    submission time.
    """
    X = df[features].copy()
    for col, dtype in category_levels().items():
        if col not in X.columns:
            continue
        unseen = sorted(set(X[col].dropna().unique()) - set(dtype.categories))
        if unseen:
            raise ValueError(
                f"{col} has levels the training data never had: {unseen}. "
                f"Decide whether to add them to the fixed set or drop the rows."
            )
        X[col] = X[col].astype(dtype)
    return X


def fit_predict(X, y, trn_idx, val_idx, params=LGBM_PARAMS) -> np.ndarray:
    model = LGBMRegressor(**params, verbose=-1)
    model.fit(X.iloc[trn_idx], y[trn_idx])
    return model.predict(X.iloc[val_idx])


def score_expanding(df, X, y, params=LGBM_PARAMS) -> list[float]:
    """RMSE per expanding-window fold. The headline score is their mean."""
    return [rmse(y[v], fit_predict(X, y, t, v, params)) for t, v in expanding_window_splits(df)]


def score_horizon(df, X, y, params=LGBM_PARAMS) -> tuple[float, pd.DataFrame]:
    """RMSE on the 16-month fold, plus the months-ahead breakdown behind it."""
    trn, val = horizon_matched_split(df)
    pred = fit_predict(X, y, trn, val, params)
    return rmse(y[val], pred), horizon_report(df, y[val], pred, val)


def expanding_run(df, X, y, *, name, features, notes, params=LGBM_PARAMS) -> dict:
    folds = score_expanding(df, X, y, params)
    return {
        "name": name, "score": float(np.mean(folds)), "fold_scores": folds,
        "validation": EXPANDING, "features": features, "model": "LGBMRegressor",
        "params": params, "notes": notes,
    }


def horizon_run(df, X, y, *, name, features, notes, params=LGBM_PARAMS) -> dict:
    """The 16-month run, with the full months-ahead table stored in the entry.

    The +13..+16 bias also goes into ``notes`` in the form the report script
    parses, so older and newer entries read the same way.
    """
    score, report = score_horizon(df, X, y, params)
    bias = report.set_index("horizon").loc["+13..+16", "bias"]
    return {
        "name": name, "score": score, "validation": HORIZON, "features": features,
        "model": "LGBMRegressor", "params": params,
        "notes": f"{notes} Bias +13..+16: {bias}.",
        "extra": {"horizon_report": report.to_dict(orient="records")},
    }


def run_baselines(df: pd.DataFrame | None = None) -> list[dict]:
    """Score the four reference runs. Returns keyword arguments for
    ``log_experiment``, one dict per run, without logging anything."""
    df = load_train() if df is None else df
    y = df[config.TARGET].to_numpy()
    X = model_matrix(df, RAW_FEATURES)
    raw = {"name": "lgbm-raw", "features": RAW_FEATURES}

    trn, val = holdout_split(df)
    median = DummyRegressor(strategy="median").fit(X.iloc[trn], y[trn]).predict(X.iloc[val])
    return [
        {
            "name": "baseline-median", "score": rmse(y[val], median), "validation": HOLDOUT,
            "features": [], "model": "DummyRegressor(median)", "params": {},
            "notes": "Floor to beat. Predicts the training median for every row.",
        },
        {
            **raw, "score": rmse(y[val], fit_predict(X, y, trn, val)), "validation": HOLDOUT,
            "model": "LGBMRegressor", "params": LGBM_PARAMS,
            "notes": "Raw columns only, no auxiliary data. Reference for quick iteration.",
        },
        expanding_run(df, X, y, **raw, notes=(
            "Raw columns only. Score is the mean of the five fold RMSEs; "
            "compare other runs fold by fold, not mean to mean."
        )),
        horizon_run(df, X, y, **raw, notes="Raw columns only, 16 months ahead."),
    ]

In [ ]:
%%module market_level
"""Rent as a market level times a relative premium, with the level forecast.

``lgbm-raw`` freezes every test prediction at the last training month, because
a tree cannot extrapolate ``MONTH_INDEX``. The EDA on main (section 7) splits
rent into two parts that behave differently:

    rent ~= market level(month) x relative premium(flat, location)

The premium is stable over time, so a tree can learn it with no time feature.
Only the level moves, and it can be forecast on its own as a 51-point monthly
series. Each arm here fits the level on training months only, trains the tree
on rent divided by that level, and multiplies back by a *forecast* level for
the months scored. Arms differ only in how the level is carried forward:

    hold     last training level, repeated -- the control. Same decomposition,
             no trend, so it separates "the split helps" from "the trend helps".
    damped   Holt's trend with damping: the recent slope, fading out.
    linear   a straight line through all training months. Expected to
             overshoot, because it averages the 2021-23 climb into the slope.
    oracle   the level fitted with the scored months' own rents. Not a
             candidate -- it sees the answers -- but it bounds what a perfect
             level forecast would buy.

The level is a hedonic index: OLS of log rent on month dummies plus flat and
location attributes, so it tracks the market rather than which flats happened
to rent that month. It is refit inside every fold.

Acceptance, fixed before running: on the 16-month fold, bias at +13..+16 moves
toward zero from -80.1 without RMSE rising above 506.18; on the expanding
window, compare fold by fold against lgbm-raw. One forecast origin cannot rank
trend methods, so a pass here earns a second look, not a default.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from . import config
from .baselines import (
    EXPANDING,
    HORIZON,
    LGBM_PARAMS,
    RAW_FEATURES,
    model_matrix,
)
from .data import load_train
from .validation import expanding_window_splits, horizon_matched_split, horizon_report, rmse

PREMIUM_FEATURES = [f for f in RAW_FEATURES if f != "MONTH_INDEX"]

# Attributes the hedonic regression holds fixed while reading off the month effect.
_HEDONIC_CATEGORICAL = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
_HEDONIC_NUMERIC = ["FLOOR_AREA_SQM", "FLAT_AGE"]

METHODS = ("hold", "damped", "linear", "oracle")


def hedonic_level(df: pd.DataFrame, idx: np.ndarray) -> pd.Series:
    """Log market level per ``MONTH_INDEX``, from the rows in ``idx`` only.

    The month coefficients of log rent on month dummies and flat attributes,
    relative to the first month. Log so that a month effect is a percentage
    move, which is what "the market rose 5%" means for every flat alike.
    """
    rows = df.iloc[idx]
    months = rows["MONTH_INDEX"]
    X = pd.concat(
        [
            pd.get_dummies(months, prefix="m", drop_first=True, dtype=float),
            pd.get_dummies(rows[_HEDONIC_CATEGORICAL], drop_first=True, dtype=float),
            rows[_HEDONIC_NUMERIC].astype(float),
        ],
        axis=1,
    )
    X.insert(0, "const", 1.0)
    coef, *_ = np.linalg.lstsq(X.to_numpy(), np.log(rows[config.TARGET].to_numpy()), rcond=None)
    coef = pd.Series(coef, index=X.columns)

    ordered = np.sort(months.unique())
    effect = [0.0] + [coef[f"m_{m}"] for m in ordered[1:]]
    return pd.Series(effect, index=ordered, name="log_level")


def forecast_level(level: pd.Series, months: np.ndarray, method: str) -> pd.Series:
    """Carry a training-months level forward to ``months`` past its end."""
    ahead = np.sort(np.unique(months[months > level.index.max()]))
    h = np.arange(1, len(ahead) + 1)
    if method == "hold":
        future = np.full(len(ahead), level.iloc[-1])
    elif method in ("damped", "blend"):
        fit = ExponentialSmoothing(
            level.to_numpy(), trend="add", damped_trend=True, initialization_method="estimated"
        ).fit()
        future = fit.forecast(len(ahead))
        if method == "blend":
            # Average of hold and damped in log space, as Part 2.3 (E6) uses.
            future = (np.full(len(ahead), level.iloc[-1]) + future) / 2
    elif method == "linear":
        slope, intercept = np.polyfit(level.index, level.to_numpy(), 1)
        future = level.iloc[-1] + slope * h
    else:
        raise ValueError(f"no forecast rule for {method!r}")
    return pd.concat([level, pd.Series(future, index=ahead)])


def fit_predict(df, X, y, trn, val, method, params=LGBM_PARAMS) -> np.ndarray:
    """Train the tree on rent / level, predict, and scale by the forecast level.

    The tree's target is a ratio, not a log: predicting in logs and
    exponentiating back returns something closer to a median than a mean,
    which biases an RMSE score low.
    """
    months = df["MONTH_INDEX"].to_numpy()
    if method == "oracle":
        level = hedonic_level(df, np.concatenate([trn, val]))
    else:
        level = forecast_level(hedonic_level(df, trn), months[val], method)
    scale = np.exp(level.reindex(months).to_numpy())

    model = LGBMRegressor(**params, verbose=-1)
    model.fit(X.iloc[trn], y[trn] / scale[trn])
    return model.predict(X.iloc[val]) * scale[val]


_NOTES = {
    "hold": "Control: level held at its last training value. Same decomposition as the "
            "trend arms with no trend, so any gain here is the split, not the forecast.",
    "damped": "Candidate: Holt's damped trend carries the level forward.",
    "linear": "Straight line through every training month. Expected to overshoot, since "
              "the slope averages in the 2021-23 climb.",
    "oracle": "Not a candidate: the level is fitted with the scored months' own rents. "
              "Bounds what a perfect level forecast could buy.",
}


def run_arms(df: pd.DataFrame | None = None) -> list[dict]:
    """Score every level method on both schemes. Returns keyword arguments for
    ``log_experiment`` without logging."""
    df = load_train() if df is None else df
    X = model_matrix(df, PREMIUM_FEATURES)
    y = df[config.TARGET].to_numpy()
    shared = {
        "features": PREMIUM_FEATURES,
        "model": "hedonic level x LGBMRegressor",
    }

    runs = []
    for method in METHODS:
        name = f"level-{method}"
        params = {**LGBM_PARAMS, "level": method}
        notes = f"Tree on rent / hedonic market level. {_NOTES[method]}"

        folds = [
            rmse(y[v], fit_predict(df, X, y, t, v, method)) for t, v in expanding_window_splits(df)
        ]
        runs.append({
            **shared, "name": name, "params": params, "notes": notes,
            "score": float(np.mean(folds)), "fold_scores": folds, "validation": EXPANDING,
        })

        trn, val = horizon_matched_split(df)
        pred = fit_predict(df, X, y, trn, val, method)
        report = horizon_report(df, y[val], pred, val)
        bias = report.set_index("horizon").loc["+13..+16", "bias"]
        runs.append({
            **shared, "name": name, "params": params,
            "notes": f"{notes} Bias +13..+16: {bias}.",
            "score": rmse(y[val], pred), "validation": HORIZON,
            "extra": {"horizon_report": report.to_dict(orient="records")},
        })
    return runs


# Second forecast origin, inside the climb: train 2021-01..2022-06, score the
# 16 months 2022-07..2023-10. The first origin (train to 2023-11) only ever
# forecast a flat market, which is the easy case for "hold". This is the case
# the damped trend is meant to insure against.
CLIMB_LAST_MONTH = 33  # 2023-10
CLIMB = "horizon-16m-origin-2022-06"


def run_climb_origin(df: pd.DataFrame | None = None) -> list[dict]:
    """``lgbm-raw`` and every level method on the 16-month fold ending 2023-10."""
    df = load_train() if df is None else df
    df = df[df["MONTH_INDEX"] <= CLIMB_LAST_MONTH].reset_index(drop=True)
    y = df[config.TARGET].to_numpy()
    trn, val = horizon_matched_split(df)

    def entry(name, features, model, params, pred, notes):
        report = horizon_report(df, y[val], pred, val)
        bias = report.set_index("horizon").loc["+13..+16", "bias"]
        return {
            "name": name, "features": features, "model": model, "params": params,
            "score": rmse(y[val], pred), "validation": CLIMB,
            "notes": f"{notes} Bias +13..+16: {bias}.",
            "extra": {"horizon_report": report.to_dict(orient="records")},
        }

    X_raw = model_matrix(df, RAW_FEATURES)
    model = LGBMRegressor(**LGBM_PARAMS, verbose=-1).fit(X_raw.iloc[trn], y[trn])
    runs = [entry(
        "lgbm-raw", RAW_FEATURES, "LGBMRegressor", LGBM_PARAMS, model.predict(X_raw.iloc[val]),
        "Reference at the climb origin: MONTH_INDEX frozen at 2022-06.",
    )]

    X = model_matrix(df, PREMIUM_FEATURES)
    for method in METHODS:
        runs.append(entry(
            f"level-{method}", PREMIUM_FEATURES, "hedonic level x LGBMRegressor",
            {**LGBM_PARAMS, "level": method}, fit_predict(df, X, y, trn, val, method),
            f"Tree on rent / hedonic market level, climb origin. {_NOTES[method]}",
        ))
    return runs

In [ ]:
%%module rent_adjustment
"""Fold-local rent adjustment shared by tabular and graph models.

Only ``fit`` reads rent labels. ``multipliers`` accepts month indices, so
forecasting validation/test scales cannot depend on their rental targets.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .config import TARGET
from .market_level import forecast_level, hedonic_level


class MarketRentAdjustment:
    """Express rents in the first fitted month's market, then restore SGD.

    ``method`` controls extrapolation of the fitted log market effects.
    Months already fitted use their observed effects, including test records
    in the final labeled month. Damped forecasts require contiguous history.
    """

    def __init__(self, method: str = "damped"):
        if method not in {"hold", "damped", "blend", "linear"}:
            raise ValueError("method must be hold, damped, blend, or linear; oracle is not a training policy")
        self.method = method

    @staticmethod
    def _months(months) -> np.ndarray:
        values = np.asarray(months)
        if values.ndim != 1 or not np.isfinite(values.astype(float)).all():
            raise ValueError("month indices must be a finite one-dimensional array")
        if not np.equal(values, values.astype(int)).all():
            raise ValueError("month indices must be integers")
        return values.astype(int)

    def fit(self, records: pd.DataFrame) -> "MarketRentAdjustment":
        if records.empty:
            raise ValueError("cannot fit a market index to empty records")
        months = self._months(records["MONTH_INDEX"])
        rents = records[TARGET].to_numpy(float)
        if not np.isfinite(rents).all() or (rents <= 0).any():
            raise ValueError("hedonic fitting requires finite positive rents")
        unique = np.unique(months)
        if self.method in ("damped", "blend") and (len(unique) < 3 or np.any(np.diff(unique) != 1)):
            raise ValueError("damped forecasting requires at least three contiguous training months")
        # Reset arbitrary/nonunique external indices before dummy construction.
        fit = records.reset_index(drop=True)
        self.log_level_ = hedonic_level(fit, np.arange(len(fit)))
        if not np.isfinite(self.log_level_.to_numpy()).all():
            raise ValueError("hedonic model produced non-finite month effects")
        self.first_month_ = int(unique[0])
        self.last_month_ = int(unique[-1])
        return self

    def multipliers(self, months) -> np.ndarray:
        """Return positive market factors in input row order, without labels."""
        if not hasattr(self, "log_level_"):
            raise RuntimeError("fit the market adjustment before transforming months")
        months = self._months(months)
        if not len(months):
            return np.empty(0, dtype=float)
        level = self.log_level_
        if months.max() > self.last_month_:
            # forecast_level counts supplied future entries as steps. Supply
            # every intervening month so sparse/repeated requests stay aligned
            # to calendar horizons, not to the number of requested months.
            future = np.arange(self.last_month_ + 1, int(months.max()) + 1)
            level = forecast_level(level, future, self.method)
        log_scale = level.reindex(months).to_numpy(float)
        if not np.isfinite(log_scale).all():
            raise ValueError("requested historical month was absent from the fitted index")
        scale = np.exp(log_scale)
        if not np.isfinite(scale).all() or (scale <= 0).any():
            raise ValueError("market forecast produced invalid multipliers")
        return scale

    def adjusted_rents(self, records: pd.DataFrame) -> np.ndarray:
        """Adjust supplied labels using the already fitted market policy."""
        return records[TARGET].to_numpy(float) / self.multipliers(records["MONTH_INDEX"])

    def restore(self, adjusted_predictions, months) -> np.ndarray:
        predictions = np.asarray(adjusted_predictions, dtype=float)
        scale = self.multipliers(months)
        if predictions.shape != scale.shape:
            raise ValueError("predictions and months must have matching one-dimensional shapes")
        return predictions * scale

## 3. Graph neural networks

The block graph, the GNN, the explainer, and the Part 2.4 experiments. Dependency order: `graph_data` → `experiment_weighted_gnns` → `compare_gnn_lightgbm` → `gnn_premiums_blend` → `gnn_neighbour_premiums` → `gnn_graph_tests`; `explain_table2_gnn` builds on `gnn_premiums_blend`.

In [ ]:
%%module block_evaluation
"""Repeatable block holdouts combined with chronological rental evaluation."""

import numpy as np
import pandas as pd


def block_time_partitions(rentals, tune_cut, outer_cut, *, fraction=0.2, seed=17):
    """Choose blocks from inner-fit history; exclude all their labels from fitting.

    A zero fraction gives the ordinary temporal control. Selection uses block IDs
    from before tune_cut only, never future activity or rental prices. Auxiliary
    graph nodes can remain visible. Returned frames preserve input row order.
    """
    if not 0 <= fraction < 1:
        raise ValueError("fraction must be in [0, 1)")
    tune_cut, outer_cut = pd.Timestamp(tune_cut), pd.Timestamp(outer_cut)
    if tune_cut >= outer_cut:
        raise ValueError("tune_cut must precede outer_cut")
    dates = pd.to_datetime(rentals["RENT_APPROVAL_DATE"])
    ids = rentals["block_id"]
    if ids.isna().any() or dates.isna().any():
        raise ValueError("Block IDs and dates must be complete")
    eligible = np.sort(ids[dates < tune_cut].unique())
    count = max(1, int(np.floor(len(eligible) * fraction))) if fraction else 0
    if fraction and count >= len(eligible):
        raise ValueError("Need at least two eligible blocks for a block holdout")
    held_ids = np.sort(np.random.default_rng(seed).choice(eligible, count, replace=False))
    retained = ~ids.isin(held_ids)
    frames = {
        "inner_fit": rentals[(dates < tune_cut) & retained].copy(),
        "inner_tune": rentals[(dates >= tune_cut) & (dates < outer_cut) & retained].copy(),
        "outer_fit": rentals[(dates < outer_cut) & retained].copy(),
        "outer_compare": rentals[dates >= outer_cut].copy(),
    }
    if any(frame.empty for frame in frames.values()):
        raise ValueError("Every temporal partition must contain rental rows")
    evaluation = frames["outer_compare"]
    fit_ids = set(frames["outer_fit"]["block_id"])
    evaluation["block_slice"] = np.where(
        evaluation["block_id"].isin(held_ids), "held_out",
        np.where(evaluation["block_id"].isin(fit_ids), "familiar", "naturally_unseen"),
    )
    return {"held_block_ids": held_ids, **frames}

In [ ]:
%%module gnn
"""Gated message-passing GNN over HDB blocks, shared by the graph scripts and notebooks.

A rental's prediction is the head applied to its block's embedding and its own record
features (floor area, flat age, flat type, flat model). The embedding comes from two
gated layers that mix in neighbouring blocks over a fixed geographic edge list.

For explanations, ``forward`` takes an optional ``edge_mask``: one value per directed
edge, multiplied into the model's own edge weights before they are normalised per
receiving block. A mask of ones reproduces the unmasked prediction; zeros remove edges.
"""

from __future__ import annotations

import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.sparse import coo_matrix
from sklearn.neighbors import BallTree
from torch import nn

EARTH_KM = 6371.0088
KINDS = ["MLP_matched", "GNN_uniform", "GNN_distance", "GNN_similarity", "GNN_learned_edges", "GNN_multiscale"]
EDGE_FEATURE_NAMES = [
    "distance_km", "completion_year_gap/20", "max_floor_gap/10",
    "same_subzone", "same_town", "nearest_mrt_distance_gap_km",
]


@dataclass
class Graph:
    """Block graph tensors. ``edge_index`` holds both directions of every edge."""

    n_nodes: int
    edge_index: torch.Tensor
    edge_features: torch.Tensor
    far_adjacency: torch.Tensor
    far_present: torch.Tensor

    @property
    def n_edges(self) -> int:
        return self.edge_index.shape[1]


def build_graph(blocks: pd.DataFrame, edges: pd.DataFrame, *, far_band_km=(0.25, 1.5),
                far_neighbors=8, far_decay_km=0.75) -> Graph:
    """Symmetric geographic edges from ``edges`` (source < target) plus the multiscale far band.

    ``blocks`` must be indexed by block_id 0..N-1 in order.
    """
    n = len(blocks)
    u = edges.source.to_numpy(int)
    v = edges.target.to_numpy(int)
    row = np.r_[u, v]
    col = np.r_[v, u]
    distance = np.tile(edges.distance_km.to_numpy(float), 2)
    year = blocks.YEAR_COMPLETED.fillna(blocks.YEAR_COMPLETED.median()).to_numpy()
    floor = blocks.MAX_FLOOR.fillna(blocks.MAX_FLOOR.median()).to_numpy()
    edge_x = np.column_stack([
        distance,
        abs(year[row] - year[col]) / 20.0,
        abs(floor[row] - floor[col]) / 10.0,
        (blocks.SUBZONE.iloc[row].to_numpy() == blocks.SUBZONE.iloc[col].to_numpy()).astype(float),
        (blocks.TOWN.iloc[row].to_numpy() == blocks.TOWN.iloc[col].to_numpy()).astype(float),
        abs(blocks.mrt_nearest_km.to_numpy()[row] - blocks.mrt_nearest_km.to_numpy()[col]),
    ]).astype("float32")

    # A separate neighbourhood band tests whether the original graph is too local.
    coords = np.radians(blocks[["LATITUDE", "LONGITUDE"]].to_numpy())
    tree = BallTree(coords, metric="haversine")
    near, far = far_band_km
    far_ids, far_dist = tree.query_radius(coords, r=far / EARTH_KM, return_distance=True, sort_results=True)
    far_rows, far_cols, far_weights = [], [], []
    for i, (ids, dists) in enumerate(zip(far_ids, far_dist)):
        km = dists * EARTH_KM
        valid = (km >= near) & (km <= far)
        ids, km = ids[valid][:far_neighbors], km[valid][:far_neighbors]
        far_rows.extend([i] * len(ids))
        far_cols.extend(ids)
        far_weights.extend(np.exp(-km / far_decay_km))
    far_sparse = coo_matrix((far_weights, (far_rows, far_cols)), shape=(n, n)).tocsr()
    far_sparse = far_sparse.maximum(far_sparse.T).tocoo()
    far_index = torch.tensor(np.stack([far_sparse.row, far_sparse.col]), dtype=torch.long)
    fw = torch.tensor(far_sparse.data, dtype=torch.float32)
    fd = torch.zeros(n).scatter_add_(0, far_index[0], fw)
    far_adjacency = torch.sparse_coo_tensor(
        far_index, fw / fd[far_index[0]].clamp_min(1e-10), (n, n), check_invariants=True
    ).coalesce()

    return Graph(
        n_nodes=n,
        edge_index=torch.tensor(np.stack([row, col]), dtype=torch.long),
        edge_features=torch.tensor(edge_x),
        far_adjacency=far_adjacency,
        far_present=fd.gt(0).float().unsqueeze(1),
    )


class WeightedRentalGNN(nn.Module):
    """Two gated message-passing layers over blocks, then a head on [block embedding, record].

    ``MLP_matched`` has the same layers with the neighbour aggregation replaced by the
    block's own embedding, so it is the graph-free control.
    """

    def __init__(self, kind: str, node_dim: int, record_dim: int, graph: Graph, hidden: int = 64):
        super().__init__()
        if kind not in KINDS:
            raise ValueError(f"unknown kind {kind!r}")
        self.kind = kind
        self.n_nodes = graph.n_nodes
        self.init_args = dict(kind=kind, node_dim=node_dim, record_dim=record_dim, hidden=hidden)
        # Parameter creation order is fixed: changing it changes every seeded run.
        self.projection = nn.Linear(node_dim, hidden)
        self.roots = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(2)])
        self.messages = nn.ModuleList([nn.Linear(hidden, hidden, bias=False) for _ in range(2)])
        self.gates = nn.ModuleList([nn.Linear(2 * hidden, hidden) for _ in range(2)])
        for gate in self.gates:
            nn.init.constant_(gate.bias, -2.0)
        self.drop = nn.Dropout(0.2)
        self.head = nn.Sequential(nn.Linear(hidden + record_dim, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
        if kind == "GNN_learned_edges":
            self.edge_network = nn.Sequential(nn.Linear(graph.edge_features.shape[1], 16), nn.ReLU(), nn.Linear(16, 1))
            nn.init.zeros_(self.edge_network[-1].weight)
            nn.init.zeros_(self.edge_network[-1].bias)
            self.log_tau = nn.Parameter(torch.tensor(float(np.log(np.expm1(0.3)))))
        if kind == "GNN_multiscale":
            self.scale_logits = nn.Parameter(torch.zeros(2))
        # Graph buffers are not saved with the weights; load_model rebuilds them from a Graph.
        self.register_buffer("far_adjacency", graph.far_adjacency, persistent=False)
        self.register_buffer("far_present", graph.far_present, persistent=False)
        self.register_buffer("edge_index", graph.edge_index, persistent=False)
        self.register_buffer("edge_features", graph.edge_features, persistent=False)
        fixed = torch.ones(graph.n_edges)
        if kind == "GNN_distance":
            fixed = torch.exp(-self.edge_features[:, 0] / 0.3)
        if kind == "GNN_similarity":
            fixed = torch.exp(
                -self.edge_features[:, 0] / 0.3
                - self.edge_features[:, 1] * (20 / 15)
                - self.edge_features[:, 2] * (10 / 8)
            )
        self.register_buffer("fixed_weights", fixed, persistent=False)
        if kind not in ["MLP_matched", "GNN_learned_edges"]:
            self.register_buffer("fixed_adjacency", self.normalize(fixed), persistent=False)

    def normalize(self, w: torch.Tensor) -> torch.Tensor:
        """Row-normalise edge weights per receiving block into a sparse adjacency."""
        degree = torch.zeros(self.n_nodes, dtype=w.dtype).scatter_add(0, self.edge_index[0], w)
        w = w / degree[self.edge_index[0]].clamp_min(1e-10)
        return torch.sparse_coo_tensor(self.edge_index, w, (self.n_nodes, self.n_nodes), check_invariants=True).coalesce()

    def edge_weights(self) -> torch.Tensor | None:
        """Raw (unnormalised) weight per directed edge, or None for the graph-free MLP."""
        if self.kind == "MLP_matched":
            return None
        if self.kind == "GNN_learned_edges":
            tau = torch.nn.functional.softplus(self.log_tau).clamp(0.03, 3.0)
            return torch.exp(-self.edge_features[:, 0] / tau) * torch.sigmoid(self.edge_network(self.edge_features).squeeze(-1))
        return self.fixed_weights

    def adjacency(self, edge_mask: torch.Tensor | None = None) -> torch.Tensor | None:
        if self.kind == "MLP_matched":
            return None
        if edge_mask is None and self.kind != "GNN_learned_edges":
            return self.fixed_adjacency
        w = self.edge_weights()
        if edge_mask is not None:
            if edge_mask.shape != w.shape:
                raise ValueError(f"edge_mask must have shape {tuple(w.shape)}, got {tuple(edge_mask.shape)}")
            w = w * edge_mask
        return self.normalize(w)

    def forward(self, nodes: torch.Tensor, part: dict, edge_mask: torch.Tensor | None = None) -> torch.Tensor:
        """Scaled premium per record in ``part`` (needs ``part['ids']`` and ``part['record']``).

        ``edge_mask`` (n_edges,) multiplies the raw edge weights; ignored by MLP_matched.
        """
        z = self.drop(torch.relu(self.projection(nodes)))
        adjacency = self.adjacency(edge_mask)
        for layer, (root, message, gate) in enumerate(zip(self.roots, self.messages, self.gates)):
            # Identity adjacency in MLP keeps the same active layers and gates.
            aggregated = z if adjacency is None else torch.sparse.mm(adjacency, z)
            if self.kind == "GNN_multiscale":
                amount = torch.sigmoid(self.scale_logits[layer]) * self.far_present
                aggregated = (1 - amount) * aggregated + amount * torch.sparse.mm(self.far_adjacency, z)
            incoming = message(aggregated)
            amount = torch.sigmoid(gate(torch.cat([z, incoming], dim=1)))
            z = self.drop(torch.relu(root(z) + amount * incoming))
        return self.head(torch.cat([z[part["ids"]], part["record"]], dim=1)).squeeze(-1)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@torch.no_grad()
def predict(model: WeightedRentalGNN, bundle: dict, part: dict, edge_mask: torch.Tensor | None = None) -> np.ndarray:
    """Rent in dollars: the model's scaled premium, un-standardised, times the market level."""
    model.eval()
    adjusted = model(bundle["nodes"], part, edge_mask).numpy() * bundle["std"] + bundle["mean"]
    return np.maximum(adjusted, 0) * part["scale"]


def train(kind: str, bundle: dict, graph: Graph, seed: int, epochs: int, *, tune: bool = False,
          patience: int = 30, hidden: int = 64, log_every: int = 40):
    """Fit on ``bundle['parts'][0]``. With ``tune``, early-stop on RMSE of ``parts[1]``.

    Returns (model, best_epoch, history). Without ``tune``, best_epoch is ``epochs``.
    """
    seed_everything(seed)
    model = WeightedRentalGNN(kind, bundle["nodes"].shape[1], bundle["parts"][0]["record"].shape[1], graph, hidden)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    fit = bundle["parts"][0]
    best = float("inf")
    best_epoch = epochs
    stale = 0
    history = []
    start = time.monotonic()
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        err = model(bundle["nodes"], fit) - fit["scaled_y"]
        loss = (fit["weight"] * err.square()).mean()
        assert torch.isfinite(loss)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        record = dict(epoch=epoch, train_objective=float(loss.detach()))
        if tune:
            val = bundle["parts"][1]
            score = float(np.sqrt(np.mean((predict(model, bundle, val) - val["y"]) ** 2)))
            record["inner_rmse"] = score
            if score < best - 0.01:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        history.append(record)
        if log_every and (epoch == 1 or epoch % log_every == 0):
            print(kind, seed, "tune" if tune else "refit", epoch, "loss", round(record["train_objective"], 4),
                  "seconds", round(time.monotonic() - start, 1), flush=True)
        if tune and stale >= patience:
            break
    return model, best_epoch, history


def save_model(model: WeightedRentalGNN, path: str | Path, **metadata) -> Path:
    """Save learned weights, constructor arguments and graph size (not the graph itself)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(dict(
        state_dict=model.state_dict(), init_args=model.init_args,
        n_nodes=model.n_nodes, n_edges=model.edge_index.shape[1], metadata=metadata,
    ), path)
    return path


def load_model(path: str | Path, graph: Graph) -> tuple[WeightedRentalGNN, dict]:
    """Rebuild a saved model on ``graph`` (must match the one it was trained on). Returns (model, metadata)."""
    saved = torch.load(path, weights_only=True)
    if (saved["n_nodes"], saved["n_edges"]) != (graph.n_nodes, graph.n_edges):
        raise ValueError(f"graph mismatch: saved {saved['n_nodes']} nodes / {saved['n_edges']} edges, "
                         f"got {graph.n_nodes} / {graph.n_edges}")
    model = WeightedRentalGNN(graph=graph, **saved["init_args"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return model, saved["metadata"]

In [ ]:
%%module explain
"""Integrated Gradients for the block GNNs, computed on each rental's exact receptive field.

A rental's prediction depends only on blocks within ``hops`` of its own block (two message-passing
layers, plus any one-hop channel read after them, stay within two hops). ``SubgraphExplainer``
cuts a trained model down to that neighbourhood by slicing its graph buffers:

- dense buffers with one row per block (e.g. ``premiums``, ``far_present``) keep the rows of the
  neighbourhood;
- sparse block x block buffers (``fixed_adjacency``, ``neighbour_adjacency``) keep the
  neighbourhood's sub-matrix. The rows of the rental's block and its direct neighbours are complete
  (all their edges stay inside a 2-hop neighbourhood), so their normalisation is unchanged; only
  the outer ring's rows are cut, and those never reach the rental;
- ``edge_index`` / ``edge_features`` / ``fixed_weights`` keep the edges inside the neighbourhood, so
  learned or masked edge weights normalise the same way for the same rows.

Integrated Gradients is then run with all steps batched as disjoint copies of the neighbourhood.
``check`` compares the cut model with the full one; notebooks assert on it.
"""

from __future__ import annotations

import copy
from dataclasses import dataclass

import numpy as np
import torch
from scipy.sparse import csr_matrix


@dataclass
class Explanation:
    """Attributions for one rental, in model output units (sum ~ f(x) - f(baseline))."""

    row: int
    block: int
    nodes: np.ndarray            # block ids of the neighbourhood, sorted
    local: int                   # position of the rental's block in ``nodes``
    record: np.ndarray           # (record_dim,)
    node: np.ndarray             # (len(nodes), node_dim)
    extra: dict                  # buffer name -> (len(nodes), width) attributions
    output: float                # model output on the neighbourhood
    baseline: float              # model output at the baseline

    @property
    def own(self) -> np.ndarray:
        return self.node[self.local]

    @property
    def neighbours(self) -> np.ndarray:
        return np.delete(self.node, self.local, axis=0)

    @property
    def neighbour_blocks(self) -> np.ndarray:
        return np.delete(self.nodes, self.local)

    def total(self) -> float:
        return float(self.record.sum() + self.node.sum() + sum(v.sum() for v in self.extra.values()))


class SubgraphExplainer:
    def __init__(self, edge_index: torch.Tensor, n_nodes: int, hops: int = 2):
        self.recv, self.send = edge_index.numpy()
        self.n_nodes = n_nodes
        self.hops = hops
        self.adjacency = csr_matrix((np.ones(len(self.recv)), (self.recv, self.send)), shape=(n_nodes, n_nodes))

    def neighbourhood(self, block: int) -> np.ndarray:
        reached = np.array([block])
        frontier = reached
        for _ in range(self.hops):
            frontier = self.adjacency[frontier].indices
            reached = np.union1d(reached, frontier)
        return reached

    def hop(self, block: int, others: np.ndarray) -> np.ndarray:
        """1 for direct neighbours of ``block``, 2 for the rest of ``others``."""
        return np.where(np.isin(others, self.adjacency[block].indices), 1, 2)

    def submodel(self, model: torch.nn.Module, nodes: np.ndarray, copies: int = 1) -> torch.nn.Module:
        if getattr(model, "kind", None) == "GNN_multiscale":
            raise ValueError("GNN_multiscale reads a 1.5 km band, beyond the 2-hop neighbourhood")
        n, total = len(nodes), len(nodes) * copies
        local = np.full(self.n_nodes, -1)
        local[nodes] = np.arange(n)
        keep = (local[self.recv] >= 0) & (local[self.send] >= 0)
        keep_t = torch.from_numpy(keep)
        offsets = torch.arange(copies) * n
        idx = torch.from_numpy(nodes)
        sub = copy.deepcopy(model).eval()
        for name, buf in list(sub.named_buffers(recurse=False)):
            if buf.is_sparse and buf.shape == (self.n_nodes, self.n_nodes):
                setattr(sub, name, self._sparse_block(buf, local, copies, n))
            elif name == "edge_index":
                ei = torch.from_numpy(np.stack([local[self.recv[keep]], local[self.send[keep]]]))
                sub.edge_index = ei.repeat(1, copies) + offsets.repeat_interleave(int(keep.sum()))
            elif name in ("edge_features", "fixed_weights"):
                part = buf[keep_t]
                setattr(sub, name, part.repeat(copies, *([1] * (part.dim() - 1))))
            elif not buf.is_sparse and buf.dim() >= 1 and buf.shape[0] == self.n_nodes:
                setattr(sub, name, buf[idx].repeat(copies, *([1] * (buf.dim() - 1))))
        sub.n_nodes = total
        return sub

    @staticmethod
    def _sparse_block(matrix: torch.Tensor, local: np.ndarray, copies: int, n: int) -> torch.Tensor:
        matrix = matrix.coalesce()
        r, c = matrix.indices().numpy()
        inside = (local[r] >= 0) & (local[c] >= 0)
        rows = torch.from_numpy(local[r[inside]])
        cols = torch.from_numpy(local[c[inside]])
        vals = matrix.values()[torch.from_numpy(inside)]
        offsets = (torch.arange(copies) * n).repeat_interleave(len(rows))
        index = torch.stack([rows.repeat(copies) + offsets, cols.repeat(copies) + offsets])
        return torch.sparse_coo_tensor(index, vals.repeat(copies), (n * copies, n * copies), check_invariants=True).coalesce()

    @torch.no_grad()
    def output(self, model, nodes_x: torch.Tensor, record: torch.Tensor, block: int, extra: dict | None = None) -> float:
        """Model output for one rental, evaluated on its neighbourhood (``extra``: full-graph buffers)."""
        S = self.neighbourhood(block)
        sub = self.submodel(model, S)
        for name, value in (extra or {}).items():
            setattr(sub, name, value[torch.from_numpy(S)])
        lb = int(np.searchsorted(S, block))
        return float(sub(nodes_x[torch.from_numpy(S)], dict(ids=torch.tensor([lb]), record=record[None]))[0])

    def explain(self, model, nodes_x: torch.Tensor, record: torch.Tensor, block: int, *, row: int,
                node_base: torch.Tensor, record_base: torch.Tensor, extra: dict | None = None,
                steps: int = 32) -> Explanation:
        """Integrated Gradients from (node_base everywhere, record_base, extra baselines) to the inputs.

        ``extra`` maps a dense per-block buffer name to (full-graph values, baseline row); those
        buffers are attributed like node features.
        """
        extra = extra or {}
        S = self.neighbourhood(block)
        lb = int(np.searchsorted(S, block))
        idx = torch.from_numpy(S)
        sub = self.submodel(model, S, copies=steps)
        alphas = ((torch.arange(steps) + 0.5) / steps)[:, None, None]
        x = nodes_x[idx]
        X = (node_base + alphas * (x - node_base)).reshape(-1, x.shape[1]).requires_grad_()
        R = (record_base + alphas[:, :, 0] * (record - record_base)).requires_grad_()
        E = {}
        for name, (values, base) in extra.items():
            v = values[idx]
            E[name] = (base + alphas * (v - base)).reshape(-1, v.shape[1]).requires_grad_()
            setattr(sub, name, E[name])
        y = sub(X, dict(ids=torch.arange(steps) * len(S) + lb, record=R))
        grads = torch.autograd.grad(y.sum(), [X, R, *E.values()])
        node_attr = (x - node_base) * grads[0].view(steps, len(S), -1).mean(0)
        rec_attr = (record - record_base) * grads[1].mean(0)
        extra_attr = {}
        for (name, (values, base)), g in zip(extra.items(), grads[2:]):
            extra_attr[name] = ((values[idx] - base) * g.view(steps, len(S), -1).mean(0)).numpy()
        with torch.no_grad():
            ones = self.submodel(model, S)
            for name, (values, base) in extra.items():
                setattr(ones, name, base.expand(len(S), -1).clone())
            baseline = float(ones(node_base.expand(len(S), -1).clone(),
                                  dict(ids=torch.tensor([lb]), record=record_base[None]))[0])
        out = self.output(model, nodes_x, record, block, {k: v for k, (v, _) in extra.items()})
        return Explanation(row=row, block=block, nodes=S, local=lb, record=rec_attr.numpy(),
                           node=node_attr.numpy(), extra=extra_attr, output=out, baseline=baseline)

In [ ]:
%%module graph_data
"""The GNN scripts' shared inputs: the block graph (committed outputs/08_graph_amenities/),
the block feature lists (outputs/09_graph_block_evaluation/config.json) and the rent frame.

Trimmed from the old scripts/investigate_graph_signal.py: its diagnostic run needed files
that are no longer in the repo, so only the loading code the GNN modules import is kept.
"""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PREVIOUS = ROOT / 'outputs/09_graph_block_evaluation'
CONFIG = json.loads((PREVIOUS / 'config.json').read_text())
TARGET = 'MONTHLY_RENT'
KEY = ['TOWN', 'BLOCK', 'STREET']
BLOCK_NUM = CONFIG['block_numeric_features']
BLOCK_CAT = CONFIG['block_categorical_features']
blocks = pd.read_csv(ROOT / 'outputs/08_graph_amenities/block_nodes.csv').sort_values('block_id').reset_index(drop=True)
assert np.array_equal(blocks.block_id, np.arange(len(blocks)))
for c in ['MAX_FLOOR', 'YEAR_COMPLETED']:
    blocks[c] = pd.to_numeric(blocks[c], errors='coerce').replace(-1, np.nan)

rent = pd.read_csv(ROOT / 'data/train.csv')
for c in KEY:
    rent[c] = rent[c].astype('string').str.strip().str.lower()
    blocks[c] = blocks[c].astype('string').str.strip().str.lower()
rent['row_id'] = np.arange(len(rent))
rent['RENT_APPROVAL_DATE'] = pd.to_datetime(rent.RENT_APPROVAL_DATE, format='%Y-%m')
rent['MONTH_INDEX'] = (rent.RENT_APPROVAL_DATE.dt.year - 2021) * 12 + rent.RENT_APPROVAL_DATE.dt.month - 1
rent['FLAT_AGE'] = rent.RENT_APPROVAL_DATE.dt.year - rent.LEASE_COMMENCE_DATE
rent['FLAT_TYPE'] = rent.FLAT_TYPE.str.lower().str.strip().str.replace(r'[\s-]+', '-', regex=True)
rent['FLAT_MODEL'] = rent.FLAT_MODEL.str.lower().str.strip()
rent = rent.merge(blocks[KEY + ['block_id']], on=KEY, how='left', validate='many_to_one', sort=False)
assert rent.row_id.tolist() == list(range(len(rent))) and rent.block_id.notna().all()
N = len(blocks)
edges = pd.read_csv(ROOT / 'outputs/08_graph_amenities/block_edges.csv')
u, v = edges.source.to_numpy(), edges.target.to_numpy()
assert np.all(u < v)


def preprocessor(num, cat):
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                          ('scale', StandardScaler())]), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat),
    ], sparse_threshold=0)


def metrics(y,pred):
    e=np.asarray(pred)-np.asarray(y)
    return dict(rmse=float(np.sqrt(np.mean(e**2))),mae=float(np.mean(abs(e))),bias=float(np.mean(e)))

In [ ]:
%%module experiment_weighted_gnns
"""Controlled GNN experiments with explicit and learned geographic edge weights.

`main()` runs them. Set the environment variable HDB_WEIGHTED_GNN_SMOKE=1 for a short execution check.
"""
# Import torch first to avoid mixed OpenMP import-order crashes.
import torch
from torch import nn
from pathlib import Path
import os,json,gc
import numpy as np
import pandas as pd
from hdb_rentals.graph_data import ROOT,blocks,rent,edges,N,BLOCK_NUM,BLOCK_CAT,preprocessor,metrics
from hdb_rentals.rent_adjustment import MarketRentAdjustment
from hdb_rentals.block_evaluation import block_time_partitions
from hdb_rentals import gnn

SMOKE=os.environ.get('HDB_WEIGHTED_GNN_SMOKE','0')=='1'
OUT=Path(os.environ['HDB_WEIGHTED_GNN_OUTPUT']) if 'HDB_WEIGHTED_GNN_OUTPUT' in os.environ else ROOT/'outputs'/('12_weighted_gnns_smoke' if SMOKE else '12_weighted_gnns')
SEEDS=[0] if SMOKE else [0,1]
KINDS=['MLP_matched','GNN_uniform','GNN_distance','GNN_similarity','GNN_learned_edges','GNN_multiscale']
if os.environ.get('HDB_WEIGHTED_GNN_KINDS'):
    requested=os.environ['HDB_WEIGHTED_GNN_KINDS'].split(',')
    assert set(requested)<=set(KINDS)
    KINDS=requested
MAX_EPOCHS=3 if SMOKE else int(os.environ.get('HDB_WEIGHTED_GNN_MAX_EPOCHS','320'))
PATIENCE=2 if SMOKE else 30
HIDDEN=64
DEVICE=torch.device('cpu')
torch.set_num_threads(int(os.environ.get('HDB_WEIGHTED_GNN_THREADS','4')))

# Graph tensors and the model live in hdb_rentals.gnn; these names stay for other scripts.
GRAPH=gnn.build_graph(blocks,edges)
EDGE_INDEX=GRAPH.edge_index;EDGE_FEATURES=GRAPH.edge_features
FAR_ADJACENCY=GRAPH.far_adjacency;FAR_PRESENT=GRAPH.far_present
WeightedRentalGNN=gnn.WeightedRentalGNN;predict=gnn.predict;seed_everything=gnn.seed_everything


def prepare(fit,*others):
    adjustment=MarketRentAdjustment('damped').fit(fit)
    bp=preprocessor(BLOCK_NUM,BLOCK_CAT+['SUBZONE'])
    bp.fit(blocks.iloc[np.sort(fit.block_id.unique())])
    node_x=torch.tensor(bp.transform(blocks).astype('float32'))
    rp=preprocessor(['FLOOR_AREA_SQM','FLAT_AGE'],['FLAT_TYPE','FLAT_MODEL']);rp.fit(fit)
    fit_scale=adjustment.multipliers(fit.MONTH_INDEX)
    adjusted=fit.MONTHLY_RENT.to_numpy()/fit_scale
    mean,std=float(adjusted.mean()),float(adjusted.std(ddof=1))
    normalizer=np.mean(fit_scale**2)
    def pack(frame):
        scale=adjustment.multipliers(frame.MONTH_INDEX)
        y=frame.MONTHLY_RENT.to_numpy(float)
        return dict(frame=frame,ids=torch.tensor(frame.block_id.to_numpy(int)),
            record=torch.tensor(rp.transform(frame).astype('float32')),
            y=y,scale=scale,scaled_y=torch.tensor(((y/scale-mean)/std).astype('float32')),
            weight=torch.tensor((scale**2/normalizer).astype('float32')))
    return dict(nodes=node_x,parts=[pack(f) for f in (fit,*others)],mean=mean,std=std)


def train(kind,bundle,seed,epochs,tune=False,save_to=None,**metadata):
    model,best_epoch,history=gnn.train(kind,bundle,GRAPH,seed,epochs,tune=tune,patience=PATIENCE,hidden=HIDDEN)
    if save_to is not None:
        gnn.save_model(model,save_to,kind=kind,seed=seed,epochs=epochs,best_epoch=best_epoch,**metadata)
    return model,best_epoch,history


def main():
    OUT.mkdir(parents=True,exist_ok=True)
    experiments=[('2023-01-01',0),('2024-01-01',0),('2025-01-01',0),('2024-01-01',17)]
    if os.environ.get('HDB_WEIGHTED_GNN_ORIGIN'):
        experiments=[job for job in experiments if job[0]==os.environ['HDB_WEIGHTED_GNN_ORIGIN'] and job[1]==0]
    if os.environ.get('HDB_WEIGHTED_GNN_SEEDS'):
        SEEDS[:]=[int(seed) for seed in os.environ['HDB_WEIGHTED_GNN_SEEDS'].split(',')]
    if SMOKE:experiments=[('2024-01-01',17)]
    config=dict(smoke=SMOKE,kinds=KINDS,seeds=SEEDS,experiments=experiments,max_epochs=MAX_EPOCHS,
        patience=PATIENCE,hidden=HIDDEN,dropout=.2,learning_rate=.001,gate_initial_bias=-2.,
        distance_decay_km=.3,similarity_age_scale_years=15,similarity_height_scale_floors=8,
        multiscale_far_band_km=[.25,1.5],multiscale_far_neighbors=8,multiscale_far_decay_km=.75,
        edge_features=['distance_km','completion_year_gap/20','max_floor_gap/10','same_subzone','same_town','nearest_mrt_distance_gap_km'],
        block_categories=BLOCK_CAT+['SUBZONE'],holdout_fraction=.2,holdout_seed=17,
        market_policy='damped',objective='SGD MSE via multiplier squared weights',
        comparison='Same active self/message/gate/head architecture; MLP uses identity adjacency',
        learned_weights='positive distance decay times learned sigmoid kernel, normalized per receiving node',
        limitations='Exploratory comparisons on previously inspected labels; metadata snapshot; fixed historical MRT policy; no actual walking distances',
        torch_version=torch.__version__,threads=torch.get_num_threads())
    (OUT/'config.json').write_text(json.dumps(config,indent=2))
    results=[];predictions=[];selections=[];histories=[];audits=[]
    for origin,split_seed in experiments:
        cutoff=pd.Timestamp(origin);inner_cut=cutoff-pd.DateOffset(months=6)
        end=min(cutoff+pd.DateOffset(months=6),rent.RENT_APPROVAL_DATE.max()+pd.DateOffset(months=1))
        partition=block_time_partitions(rent,inner_cut,cutoff,fraction=.2 if split_seed else 0,seed=split_seed)
        fit=partition['outer_fit'];val=partition['outer_compare']
        val=val[val.RENT_APPROVAL_DATE<end].copy()
        inner_fit=partition['inner_fit'];inner_val=partition['inner_tune']
        held=set(partition['held_block_ids'])
        for frame in [fit,inner_fit,inner_val]:assert not frame.block_id.isin(held).any()
        if SMOKE:
            fit=fit.sample(min(3500,len(fit)),random_state=0).sort_index()
            inner_fit=inner_fit.sample(min(2500,len(inner_fit)),random_state=0).sort_index()
            inner_val=inner_val.sample(min(800,len(inner_val)),random_state=0).sort_index()
            val=pd.concat([g.sample(min(300,len(g)),random_state=0) for _,g in val.groupby('block_slice')]).sort_index()
        experiment=f'{cutoff:%Y%m}_'+('temporal' if not split_seed else f'held_{split_seed}')
        print('EXPERIMENT',experiment,'fit',len(fit),'evaluate',len(val),flush=True)
        pd.DataFrame({'block_id':np.arange(N),'held_out':np.isin(np.arange(N),list(held))}).to_csv(OUT/f'{experiment}_block_membership.csv',index=False)
        inner=prepare(inner_fit,inner_val);outer=prepare(fit,val)
        audits.append(dict(experiment=experiment,inner_fit_max=str(inner_fit.RENT_APPROVAL_DATE.max()),
            inner_val_min=str(inner_val.RENT_APPROVAL_DATE.min()),outer_fit_max=str(fit.RENT_APPROVAL_DATE.max()),
            outer_val_min=str(val.RENT_APPROVAL_DATE.min()),held_block_count=len(held),held_training_rows=0))
        for kind in KINDS:
            for seed in SEEDS:
                tags=dict(experiment=experiment,origin=origin,split_seed=split_seed,model=kind,seed=seed)
                tuned,count,history=train(kind,inner,seed,MAX_EPOCHS,tune=True)
                histories.extend([{**tags,**h} for h in history]);del tuned;gc.collect()
                fitted,_,_=train(kind,outer,seed,count,save_to=OUT/'models'/f'{experiment}_{kind}_seed{seed}.pt',
                                 experiment=experiment,origin=origin,split_seed=split_seed)
                pred=predict(fitted,outer,outer['parts'][1])
                frame=val[['row_id','block_id','RENT_APPROVAL_DATE','MONTHLY_RENT','block_slice']].copy()
                frame['prediction']=pred
                for name,value in tags.items():frame[name]=value
                predictions.append(frame)
                for slice_name,g in [('all',frame)]+list(frame.groupby('block_slice')):
                    results.append(dict(**tags,slice=slice_name,n_rows=len(g),n_blocks=g.block_id.nunique(),
                        **metrics(g.MONTHLY_RENT,g.prediction)))
                selections.append(dict(**tags,iterations=count,parameters=sum(p.numel() for p in fitted.parameters()),
                                       selected_inner_rmse=min(h['inner_rmse'] for h in history)))
                if kind=='GNN_learned_edges':
                    adj=fitted.adjacency().coalesce()
                    idx=adj.indices().numpy();weight=adj.values().detach().numpy()
                    pd.DataFrame(dict(source=idx[0],target=idx[1],normalized_weight=weight)).to_csv(
                        OUT/f'{experiment}_seed{seed}_learned_weights.csv',index=False)
                print('RESULT',tags,'epochs',count,metrics(val.MONTHLY_RENT,pred),flush=True)
                del fitted;gc.collect()
                pd.DataFrame(results).to_csv(OUT/'seed_metrics.csv',index=False)
                pd.DataFrame(selections).to_csv(OUT/'selected_iterations.csv',index=False)
                pd.DataFrame(histories).to_csv(OUT/'tuning_history.csv',index=False)
                pd.DataFrame(audits).to_csv(OUT/'isolation_audit.csv',index=False)
        del inner,outer;gc.collect()
        pd.concat(predictions,ignore_index=True).to_csv(OUT/'predictions.csv',index=False)
    print('WEIGHTED GNN EXPERIMENTS COMPLETE',flush=True)

In [ ]:
%%module compare_gnn_lightgbm
"""GNN vs Part 2.3 on Part 2.3's own splits, rows and (optionally) block features.

    main()               # full run
    main(["--smoke"])    # 3 epochs, a quick execution check

Windows are Part 2.3:
    tuning   fit 2021-01..2023-03, score 2023-04..2023-12 -> GNN early stopping (epoch count)
    holdout  fit 2021-01..2023-12, score 2024-01..2025-03 -> the reported RMSE
The GNN is refit on the holdout fit rows for the epoch count chosen on the tuning split,
as experiment_weighted_gnns.py does. Notebook 10's E0 LightGBM is re-run here first; it must
reproduce the notebook's 603.6 / 487.5, which checks that both sides score the same rows.

Arms (one seed each, as Part 2.3 reports one LightGBM run):
    level     how rent / market level is carried past the last fit month: damped (the GNN
              scripts' default) or blend (Part 2.3's E6 choice)
    features  gnn = the GNN's block table as-is; part23 = plus Part 2.3's block columns the
              GNN lacks (hawker, CBD distance, MRT interchange)
    model     GNN_uniform, and MLP_matched (same network with the graph switched off)
Notebook 10's block / street premiums are not added; compare with E0, not E5/S1.
"""
import argparse
import json
import sys
import time
from pathlib import Path


# Import torch (via the GNN module) before LightGBM, as experiment_weighted_gnns.py notes.
from hdb_rentals import experiment_weighted_gnns as ewg
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from hdb_rentals.rent_adjustment import MarketRentAdjustment

OUT = ROOT / "outputs" / "gnn_vs_lightgbm"
DATA = ROOT / "data"
CLEANED = DATA / "cleaned"
TARGET, KEY = "MONTHLY_RENT", ["TOWN", "BLOCK", "STREET"]
TUNE_START, HOLD_START, HOLD_END = pd.Timestamp("2023-04-01"), pd.Timestamp("2024-01-01"), pd.Timestamp("2025-04-01")

# ---------------------------------------------------------------- Part 2.3
# Copied, not imported, so these numbers are built exactly as the notebook builds them.


def lower_strip(df):
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df):
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    d = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = d
    out["MONTH_INDEX"] = (d.dt.year - 2021) * 12 + d.dt.month - 1
    return out


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def part23_block_table():
    hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
    mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
    schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
    malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))
    with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
        geo = json.load(f)["features"]
    hawker = lower_strip(pd.DataFrame({
        "NAME": [g["properties"]["NAME"] for g in geo],
        "STATUS": [g["properties"]["STATUS"] for g in geo],
        "LONGITUDE": [g["geometry"]["coordinates"][0] for g in geo],
        "LATITUDE": [g["geometry"]["coordinates"][1] for g in geo],
    }))
    mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
    poi_groups = {
        "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
        "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
        "mrt": mrt_open,
        "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
        "malls": malls,
        "hawker": hawker[hawker["STATUS"].ne("under construction")],
    }
    lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
    for prefix, poi in poi_groups.items():
        d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
        hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
        for label, r in {"1km": 1.0, "2km": 2.0}.items():
            hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
    cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
    hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])
    return hdb


DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]
# The DIST columns the GNN's block table has no counterpart for.
PART23_EXTRA = ["hawker_nearest_km", "hawker_within_1km", "dist_to_cbd_km",
              "mrt_interchange_nearest_km", "mrt_interchange_within_1km"]

# ---------------------------------------------------------------- Part 2.3: E0
LGBM_TUNED = {"num_leaves": 63, "min_child_samples": 100, "subsample": 0.85, "subsample_freq": 1,
              "colsample_bytree": 0.5, "reg_lambda": 20.0, "cat_smooth": 30.0, "n_estimators": 357,
              "learning_rate": 0.05}


def part23_e0(hdb):
    tr = clean_rent(pd.read_csv(DATA / "train.csv")).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
    tr = tr.merge(hdb[KEY + DIST + ["LATITUDE", "LONGITUDE"]], on=KEY, how="left", validate="m:1")
    num, cat = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"] + DIST, ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
    categories = {c: pd.CategoricalDtype(sorted(tr[c].unique())) for c in cat}

    def matrix(df):
        X = df[num + cat].copy()
        for c in cat:
            X[c] = X[c].astype(categories[c])
        return X

    rows = []
    for split, start, end in [("tuning", TUNE_START, HOLD_START), ("holdout", HOLD_START, HOLD_END)]:
        d = tr["RENT_APPROVAL_DATE"]
        fit_df, val_df = tr[d < start], tr[(d >= start) & (d < end)]
        est = lgb.LGBMRegressor(**LGBM_TUNED, verbose=-1, n_jobs=-1, random_state=0)
        p = est.fit(matrix(fit_df), fit_df[TARGET].to_numpy()).predict(matrix(val_df))
        rows.append(dict(split=split, model="part23_E0_LightGBM", level="MONTH_INDEX", features="part23",
                         n_rows=len(val_df), **ewg.metrics(val_df[TARGET], p)))
    return rows

# ---------------------------------------------------------------- GNN side


def prepare(fit, others, level, block_num, record_extra=()):
    # experiment_weighted_gnns.prepare with the level method, block columns and extra row
    # columns as arguments.
    adjustment = MarketRentAdjustment(level).fit(fit)
    bp = ewg.preprocessor(block_num, ewg.BLOCK_CAT + ["SUBZONE"])
    bp.fit(ewg.blocks.iloc[np.sort(fit.block_id.unique())])
    node_x = torch.tensor(bp.transform(ewg.blocks).astype("float32"))
    rp = ewg.preprocessor(["FLOOR_AREA_SQM", "FLAT_AGE", *record_extra], ["FLAT_TYPE", "FLAT_MODEL"])
    rp.fit(fit)
    fit_scale = adjustment.multipliers(fit.MONTH_INDEX)
    adjusted = fit.MONTHLY_RENT.to_numpy() / fit_scale
    mean, std = float(adjusted.mean()), float(adjusted.std(ddof=1))
    normalizer = np.mean(fit_scale ** 2)

    def pack(frame):
        scale = adjustment.multipliers(frame.MONTH_INDEX)
        y = frame.MONTHLY_RENT.to_numpy(float)
        return dict(frame=frame, ids=torch.tensor(frame.block_id.to_numpy(int)),
                    record=torch.tensor(rp.transform(frame).astype("float32")),
                    y=y, scale=scale, scaled_y=torch.tensor(((y / scale - mean) / std).astype("float32")),
                    weight=torch.tensor((scale ** 2 / normalizer).astype("float32")))
    return dict(nodes=node_x, parts=[pack(f) for f in (fit, *others)], mean=mean, std=std)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true", help="3 epochs, patience 2, small row samples")
    ap.add_argument("--levels", default="damped,blend")
    ap.add_argument("--features", default="gnn,part23")
    ap.add_argument("--models", default="MLP_matched,GNN_uniform")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--max-epochs", type=int, default=320)
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args(argv)
    max_epochs = 3 if args.smoke else args.max_epochs
    ewg.PATIENCE = 2 if args.smoke else 30
    torch.set_num_threads(args.threads)
    out = OUT / "smoke" if args.smoke else OUT
    out.mkdir(parents=True, exist_ok=True)

    # Notebook 10's block columns onto the GNN's block table, by town + block + street.
    hdb = part23_block_table()
    blocks = ewg.blocks
    extra = lower_strip(hdb[KEY + PART23_EXTRA])
    merged = blocks[KEY].merge(extra, on=KEY, how="left", validate="m:1")
    assert len(merged) == len(blocks)
    for c in PART23_EXTRA:
        blocks[c] = merged[c].to_numpy(float)
    rented = blocks.block_id.isin(ewg.rent.block_id)
    print(f"part23 columns matched for {blocks.loc[rented, PART23_EXTRA[0]].notna().mean():.1%} of rented blocks, "
          f"{blocks[PART23_EXTRA[0]].notna().mean():.1%} of all {len(blocks):,} graph blocks", flush=True)
    feature_sets = {"gnn": list(ewg.BLOCK_NUM), "part23": list(ewg.BLOCK_NUM) + PART23_EXTRA}

    results = part23_e0(hdb)
    for r in results:
        print("REFERENCE", r, flush=True)

    rent = ewg.rent
    d = rent.RENT_APPROVAL_DATE
    inner_fit, inner_val = rent[d < TUNE_START], rent[(d >= TUNE_START) & (d < HOLD_START)]
    fit, val = rent[d < HOLD_START], rent[(d >= HOLD_START) & (d < HOLD_END)]
    if args.smoke:
        inner_fit, fit = inner_fit.sample(2500, random_state=0).sort_index(), fit.sample(3500, random_state=0).sort_index()
        inner_val, val = inner_val.sample(800, random_state=0).sort_index(), val.sample(800, random_state=0).sort_index()
    print(f"GNN rows: tuning fit {len(inner_fit):,} score {len(inner_val):,}; "
          f"holdout fit {len(fit):,} score {len(val):,}", flush=True)

    config = dict(vars(args), max_epochs=max_epochs, patience=ewg.PATIENCE, hidden=ewg.HIDDEN,
                  tuning=[str(TUNE_START.date()), str(HOLD_START.date())],
                  holdout=[str(HOLD_START.date()), str(HOLD_END.date())], part23_extra=PART23_EXTRA,
                  torch_version=torch.__version__)
    (out / "config.json").write_text(json.dumps(config, indent=2))

    predictions = []
    for level in args.levels.split(","):
        for feat in args.features.split(","):
            inner = prepare(inner_fit, [inner_val], level, feature_sets[feat])
            outer = prepare(fit, [val], level, feature_sets[feat])
            for kind in args.models.split(","):
                tags = dict(model=kind, level=level, features=feat, seed=args.seed)
                start = time.monotonic()
                tuned, epochs, history = ewg.train(kind, inner, args.seed, max_epochs, tune=True)
                tune_rmse = min(h["inner_rmse"] for h in history)
                fitted, _, _ = ewg.train(kind, outer, args.seed, epochs)
                pred = ewg.predict(fitted, outer, outer["parts"][1])
                results.append(dict(split="tuning", **tags, epochs=epochs, n_rows=len(inner_val), rmse=tune_rmse))
                results.append(dict(split="holdout", **tags, epochs=epochs, n_rows=len(val),
                                    **ewg.metrics(val.MONTHLY_RENT, pred)))
                frame = val[["row_id", "block_id", "RENT_APPROVAL_DATE", "MONTHLY_RENT"]].copy()
                frame["prediction"] = pred
                predictions.append(frame.assign(**tags))
                print("RESULT", tags, "epochs", epochs, "tuning", round(tune_rmse, 2),
                      "holdout", ewg.metrics(val.MONTHLY_RENT, pred),
                      "minutes", round((time.monotonic() - start) / 60, 1), flush=True)
                pd.DataFrame(results).to_csv(out / "results.csv", index=False)
                pd.concat(predictions, ignore_index=True).to_csv(out / "predictions.csv", index=False)

    r = pd.DataFrame(results)
    arms = r[["model", "level", "features"]].drop_duplicates()
    table = r.set_index(["model", "level", "features", "split"])["rmse"].unstack("split")
    table = table.reindex(pd.MultiIndex.from_frame(arms))[["tuning", "holdout"]]
    table["holdout vs E0"] = table["holdout"] - table.loc[("part23_E0_LightGBM", "MONTH_INDEX", "part23"), "holdout"]
    print("\nRMSE (SGD), Part 2.3 windows\n", table.round(2).to_string(), flush=True)
    table.round(2).to_csv(out / "summary.csv")

In [ ]:
%%module gnn_premiums_blend
"""GNN with Part 2.3's block premiums, and GNN + LightGBM blends, on Part 2.3's windows.

    main()                                       # full run
    main(["--smoke"])                            # 3 epochs, a quick execution check
    main(["--level", "blend", "--same-level"])   # both models on one level

Same windows, rows, level (damped) and block columns (the GNN's own) as compare_gnn_lightgbm.
--level sets the neural model's market level (damped or blend). By default LightGBM keeps
`MONTH_INDEX`, as in Part 2.3's E0/E1; --same-level makes it learn rent / the same level instead
(Part 2.3's E6 setup: `MONTH_INDEX` dropped, rows weighted by level squared), so the two models
differ only in the premium model. Non-default runs write to a subfolder named after the options.

Premium inputs (Part 2.3 E1: block and block + flat type premiums of relative rent, smoothing
20 / 5, plus the block + flat type row count; built from each split's fit rows only):
    none      no premiums (compare_gnn_lightgbm's arm, re-run for its tuning predictions)
    row       each row gets E1's premiums: out-of-fold for fit rows, all fit rows for scored rows,
              exactly the columns Part 2.3's LightGBM gets
    row+node  as row, and each block node also carries its block premium and log row count, so
              the graph can pass premiums between neighbouring blocks. Node premiums use all fit
              rows (a fit row's own rent is in its block's node), so this arm may overfit; scored
              rows' rents never enter either way, and the tuning split decides whether it is kept.

Blends: prediction = w * neural + (1 - w) * LightGBM, with w chosen on the tuning split (grid 0.05)
and applied unchanged to the holdout. The partner is E0 for `none` and E1 for the premium arms.
The neural model's tuning predictions come from a refit on the tuning fit rows for the chosen
epoch count, the same model the early stopping picked.
"""
import argparse
import json
import sys
import time
from pathlib import Path


from hdb_rentals import compare_gnn_lightgbm as cmp  # (imports torch before LightGBM)
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold

ewg = cmp.ewg
OUT = ROOT / "outputs" / "gnn_premiums_blend"
TARGET, KEY = cmp.TARGET, cmp.KEY
PREM = ["BLOCK_PREM", "BLOCK_FT_PREM", "BLOCK_FT_N"]
NODE_PREM = ["NODE_BLOCK_PREM", "NODE_LOG_N"]
M_BLOCK, M_FT = 20, 5
SPLITS = {"tuning": (cmp.TUNE_START, cmp.HOLD_START), "holdout": (cmp.HOLD_START, cmp.HOLD_END)}

# ---------------------------------------------------------------- Part 2.3


def part23_premium_frames(hdb):
    """Notebook 10's frames per split with E1 `oof` premiums, plus block-level fit premiums."""
    raw = cmp.clean_rent(pd.read_csv(cmp.DATA / "train.csv"))
    raw["row_id"] = np.arange(len(raw))  # train.csv order, as the GNN's rent frame
    tr = raw.sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
    te = cmp.clean_rent(pd.read_csv(cmp.DATA / "test.csv"))
    tr = tr.merge(hdb[KEY + cmp.DIST], on=KEY, how="left", validate="m:1")
    blocks = pd.concat([tr, te])[KEY].drop_duplicates().reset_index(drop=True)
    blocks["BID"] = np.arange(len(blocks))
    ft_levels = sorted(tr["FLAT_TYPE"].unique())
    nb, nft = len(blocks), len(ft_levels)
    tr = tr.merge(blocks, on=KEY, how="left", validate="m:1")
    tr["FT"] = pd.Categorical(tr["FLAT_TYPE"], categories=ft_levels).codes
    tr["G"] = tr["BID"] * nft + tr["FT"]
    tr["REL"] = tr[TARGET] / tr.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median")

    def encode(src, dst):
        bid, g, rel = src["BID"].to_numpy(), src["G"].to_numpy(), src["REL"].to_numpy()
        w = np.ones(len(src))
        sb, cb = np.bincount(bid, w * rel, nb), np.bincount(bid, w, nb)
        sg, cg = np.bincount(g, w * rel, nb * nft), np.bincount(g, w, nb * nft)
        n = np.bincount(g, minlength=nb * nft)
        db, dg = dst["BID"].to_numpy(), dst["G"].to_numpy()
        pb = (M_BLOCK + sb[db]) / (M_BLOCK + cb[db])
        pg = (M_FT * pb + sg[dg]) / (M_FT + cg[dg])
        return np.column_stack([pb, pg, n[dg]])

    frames = {}
    for split, (start, end) in SPLITS.items():
        d = tr["RENT_APPROVAL_DATE"]
        fit_df = tr[d < start].reset_index(drop=True)
        val_df = tr[(d >= start) & (d < end)].reset_index(drop=True)
        own = np.empty((len(fit_df), 3))
        for f_idx, p_idx in KFold(5, shuffle=True, random_state=0).split(fit_df):
            own[p_idx] = encode(fit_df.iloc[f_idx], fit_df.iloc[p_idx])
        fit_df[PREM], val_df[PREM] = own, encode(fit_df, val_df)
        sb = np.bincount(fit_df["BID"], fit_df["REL"], nb)
        cb = np.bincount(fit_df["BID"], minlength=nb).astype(float)
        node = blocks[KEY].assign(NODE_BLOCK_PREM=(M_BLOCK + sb) / (M_BLOCK + cb), NODE_LOG_N=np.log1p(cb))
        frames[split] = (fit_df, val_df, node)
    return tr, frames


def part23_lgbm(tr, frames, level=None):
    """E0 and E1 (E0 + `oof` premiums) LightGBM, predictions keyed by row_id.

    level=None keeps `MONTH_INDEX` (Part 2.3's E0/E1). A level method instead fits rent / level
    without `MONTH_INDEX`, weighted by level squared (Part 2.3's E6)."""
    time_col = [] if level else ["MONTH_INDEX"]
    num, cat = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE"] + time_col + cmp.DIST, ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
    categories = {c: pd.CategoricalDtype(sorted(tr[c].unique())) for c in cat}

    def matrix(df, extra):
        X = df[num + list(extra) + cat].copy()
        for c in cat:
            X[c] = X[c].astype(categories[c])
        return X

    preds, rows = {}, []
    for name, extra in [("part23_E0_LightGBM", []), ("part23_E1_LightGBM", PREM)]:
        for split, (fit_df, val_df, _) in frames.items():
            est = lgb.LGBMRegressor(**cmp.LGBM_TUNED, verbose=-1, n_jobs=-1, random_state=0)
            if level:
                # Fitted on the GNN's rent frame (same rows, has FLAT_AGE), as cmp.prepare does.
                adj = cmp.MarketRentAdjustment(level).fit(ewg.rent[ewg.rent.row_id.isin(fit_df.row_id)])
                fs, vs = adj.multipliers(fit_df["MONTH_INDEX"]), adj.multipliers(val_df["MONTH_INDEX"])
                est.fit(matrix(fit_df, extra), fit_df[TARGET].to_numpy() / fs, sample_weight=fs ** 2)
                p = est.predict(matrix(val_df, extra)) * vs
            else:
                p = est.fit(matrix(fit_df, extra), fit_df[TARGET].to_numpy()).predict(matrix(val_df, extra))
            preds[(name, split)] = pd.Series(p, index=val_df["row_id"].to_numpy())
            rows.append(dict(split=split, model=name, premiums="row" if extra else "none",
                             n_rows=len(val_df), **ewg.metrics(val_df[TARGET], p)))
    return preds, rows

# ---------------------------------------------------------------- GNN side


def with_premiums(frame, prem_frame):
    out = frame.merge(prem_frame[["row_id"] + PREM], on="row_id", how="left", validate="1:1")
    assert out[PREM].notna().all().all() and out["row_id"].tolist() == frame["row_id"].tolist()
    out["BLOCK_FT_N"] = np.log1p(out["BLOCK_FT_N"])  # counts on a log scale for the network
    return out


def best_blend(y_tune, nn_tune, gb_tune):
    grid = np.round(np.arange(0, 1.0001, 0.05), 2)
    scores = [ewg.metrics(y_tune, w * nn_tune + (1 - w) * gb_tune)["rmse"] for w in grid]
    return float(grid[int(np.argmin(scores))])


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--premiums", default="none,row,row+node")
    ap.add_argument("--models", default="MLP_matched,GNN_uniform")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--max-epochs", type=int, default=320)
    ap.add_argument("--threads", type=int, default=4)
    ap.add_argument("--level", default="damped", choices=["damped", "blend", "hold"])
    ap.add_argument("--same-level", action="store_true", help="LightGBM uses the neural model's level")
    args = ap.parse_args(argv)
    max_epochs = 3 if args.smoke else args.max_epochs
    ewg.PATIENCE = 2 if args.smoke else 30
    torch.set_num_threads(args.threads)
    out = OUT / "smoke" if args.smoke else OUT
    if args.level != "damped" or args.same_level or args.seed != 0:
        out = out / f"level-{args.level}{'_same' if args.same_level else ''}_seed{args.seed}"
    out.mkdir(parents=True, exist_ok=True)

    hdb = cmp.part23_block_table()
    tr, frames = part23_premium_frames(hdb)
    gb_preds, results = part23_lgbm(tr, frames, args.level if args.same_level else None)
    for r in results:
        print("REFERENCE", r, flush=True)

    rent, blocks = ewg.rent, ewg.blocks
    d = rent.RENT_APPROVAL_DATE
    parts = {"tuning": (rent[d < cmp.TUNE_START], rent[(d >= cmp.TUNE_START) & (d < cmp.HOLD_START)]),
             "holdout": (rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)])}
    for split, (fit_df, val_df, _) in frames.items():
        assert set(parts[split][0].row_id) == set(fit_df.row_id) and set(parts[split][1].row_id) == set(val_df.row_id)
    parts = {s: tuple(with_premiums(f, pd.concat(frames[s][:2])) for f in pair) for s, pair in parts.items()}
    if args.smoke:
        parts = {s: (f.sample(3000, random_state=0).sort_index(), v.sample(800, random_state=0).sort_index())
                 for s, (f, v) in parts.items()}
    # Graph blocks never rented in train or test get the no-history premium (1, zero rows).
    node_prem = {s: blocks[KEY].merge(frames[s][2], on=KEY, how="left", validate="m:1")[NODE_PREM]
                 .fillna({"NODE_BLOCK_PREM": 1.0, "NODE_LOG_N": 0.0}).to_numpy(float) for s in SPLITS}
    print(f"graph blocks with no rentals in train or test: "
          f"{blocks[KEY].merge(frames['holdout'][2], on=KEY, how='left')['NODE_LOG_N'].isna().sum()}", flush=True)

    (out / "config.json").write_text(json.dumps(dict(vars(args), max_epochs=max_epochs, patience=ewg.PATIENCE,
                                                     level=args.level, smoothing=[M_BLOCK, M_FT]), indent=2))
    predictions = []
    for prem in args.premiums.split(","):
        record_extra = [] if prem == "none" else PREM
        block_num = list(ewg.BLOCK_NUM) + (NODE_PREM if prem == "row+node" else [])
        bundles = {}
        for s, (f, v) in parts.items():
            if prem == "row+node":
                blocks[NODE_PREM] = node_prem[s]  # read by prepare when it builds node features
            bundles[s] = cmp.prepare(f, [v], args.level, block_num, record_extra)
        partner = "part23_E0_LightGBM" if prem == "none" else "part23_E1_LightGBM"
        for kind in args.models.split(","):
            tags = dict(model=kind, premiums=prem, seed=args.seed)
            start = time.monotonic()
            _, epochs, history = ewg.train(kind, bundles["tuning"], args.seed, max_epochs, tune=True)
            nn = {}
            for s in SPLITS:
                fitted, _, _ = ewg.train(kind, bundles[s], args.seed, epochs)
                nn[s] = pd.Series(ewg.predict(fitted, bundles[s], bundles[s]["parts"][1]),
                                  index=parts[s][1]["row_id"].to_numpy())
            y = {s: parts[s][1].set_index("row_id")[TARGET] for s in SPLITS}
            gb = {s: gb_preds[(partner, s)].reindex(y[s].index) for s in SPLITS}
            w = best_blend(y["tuning"], nn["tuning"], gb["tuning"])
            for s in SPLITS:
                results.append(dict(split=s, **tags, epochs=epochs, n_rows=len(y[s]), **ewg.metrics(y[s], nn[s])))
                results.append(dict(split=s, model=f"{kind} + {partner.split('_')[1]} blend", premiums=prem,
                                    seed=args.seed, weight=w, n_rows=len(y[s]),
                                    **ewg.metrics(y[s], w * nn[s] + (1 - w) * gb[s])))
                predictions.append(pd.DataFrame({"row_id": y[s].index, "split": s, TARGET: y[s].to_numpy(),
                                                 "neural": nn[s].to_numpy(), "lightgbm": gb[s].to_numpy(), **tags}))
            hold = [r for r in results[-4:] if r["split"] == "holdout"]
            print("RESULT", tags, "epochs", epochs, "holdout", round(hold[0]["rmse"], 2),
                  f"| blend w={w}", "holdout", round(hold[1]["rmse"], 2),
                  "minutes", round((time.monotonic() - start) / 60, 1), flush=True)
            pd.DataFrame(results).to_csv(out / "results.csv", index=False)
            pd.concat(predictions, ignore_index=True).to_csv(out / "predictions.csv", index=False)

    r = pd.DataFrame(results)
    arms = r[["model", "premiums"]].drop_duplicates()
    table = r.set_index(["model", "premiums", "split"])["rmse"].unstack("split")
    table = table.reindex(pd.MultiIndex.from_frame(arms))[["tuning", "holdout"]]
    table["weight"] = r[r.split == "holdout"].set_index(["model", "premiums"])["weight"].reindex(table.index) \
        if "weight" in r else np.nan
    for ref in ["E0", "E1"]:
        table[f"holdout vs {ref}"] = table["holdout"] - table.loc[(f"part23_{ref}_LightGBM", "none" if ref == "E0" else "row"), "holdout"]
    print("\nRMSE (SGD), Part 2.3 windows\n", table.round(2).to_string(), flush=True)
    table.round(2).to_csv(out / "summary.csv")

In [ ]:
%%module gnn_neighbour_premiums
"""Neighbours' premiums through the graph without leakage, and errors by how much history a block has.

    main()               # full run
    main(["--smoke"])    # 3 epochs, a quick execution check

Same windows, rows, level (damped), block columns and row premiums (`row`, Part 2.3 E1) as
gnn_premiums_blend. New here:

Neighbour premium channel (`row+nbr`). gnn_premiums_blend's `row+node` put each block's premium,
built from all fit rows, into its own node, so a fit row could read its own rent. Here node
premiums enter only through a separate one-hop channel added after the last message-passing layer:

    h_i = mean over neighbours j of f(premium_j, log rows_j)        (graph has no self-loops)
    z_i <- z_i + gate(z_i, h_i) * W h_i

A block never receives its own premium, and with one hop it cannot come back through a neighbour
(two layers of ordinary message passing would return it). The channel is added to both GNN_uniform
and MLP_matched; the MLP version isolates the neighbour premiums from ordinary message passing.

Breakdown. Holdout RMSE of every model by how many fit rows its block + flat type has (0, 1-2, 3-5,
6-10, >10), the slice where borrowing from neighbours should matter. `row` arms are read from
outputs/gnn_premiums_blend/predictions.csv, so run gnn_premiums_blend first.
"""
import argparse
import json
import sys
import time
from pathlib import Path


from hdb_rentals import gnn_premiums_blend as gpb  # (imports torch before LightGBM)
import numpy as np
import pandas as pd
import torch
from torch import nn

cmp, ewg = gpb.cmp, gpb.ewg
OUT = ROOT / "outputs" / "gnn_neighbour_premiums"
TARGET, KEY = cmp.TARGET, cmp.KEY
BUCKETS = ([-1, 0, 2, 5, 10, np.inf], ["0", "1-2", "3-5", "6-10", ">10"])


class NeighbourPremiumGNN(ewg.WeightedRentalGNN):
    """WeightedRentalGNN plus a one-hop channel carrying neighbours' (never own) premiums."""

    def __init__(self, kind, node_dim, record_dim, premiums):
        super().__init__(kind, node_dim, record_dim, ewg.GRAPH, hidden=ewg.HIDDEN)
        hidden = ewg.HIDDEN
        self.register_buffer("premiums", premiums)
        # Uniform mean over neighbours; edge_index holds both directions and no self-loops.
        self.register_buffer("neighbour_adjacency", self.normalize(torch.ones(self.edge_index.shape[1])))
        self.premium_in = nn.Sequential(nn.Linear(premiums.shape[1], hidden), nn.ReLU())
        self.premium_message = nn.Linear(hidden, hidden, bias=False)
        self.premium_gate = nn.Linear(2 * hidden, hidden)
        nn.init.constant_(self.premium_gate.bias, -2.)

    def forward(self, nodes, part, edge_mask=None):
        # WeightedRentalGNN.forward (uniform / MLP kinds), then the premium channel, then the head.
        # edge_mask multiplies each directed edge in message passing and in the premium channel.
        z = self.drop(torch.relu(self.projection(nodes)))
        adjacency = self.adjacency(edge_mask)
        for root, message, gate in zip(self.roots, self.messages, self.gates):
            aggregated = z if adjacency is None else torch.sparse.mm(adjacency, z)
            incoming = message(aggregated)
            amount = torch.sigmoid(gate(torch.cat([z, incoming], dim=1)))
            z = self.drop(torch.relu(root(z) + amount * incoming))
        neighbours = self.neighbour_adjacency if edge_mask is None else self.normalize(edge_mask)
        h = self.premium_message(torch.sparse.mm(neighbours, self.premium_in(self.premiums)))
        amount = torch.sigmoid(self.premium_gate(torch.cat([z, h], dim=1)))
        self.last_premium_gate = amount.detach()
        z = z + amount * h
        return self.head(torch.cat([z[part["ids"]], part["record"]], dim=1)).squeeze(-1)


def train(kind, bundle, seed, epochs, tune=False):
    # ewg.train, building NeighbourPremiumGNN with the bundle's node premiums.
    ewg.seed_everything(seed)
    model = NeighbourPremiumGNN(kind, bundle["nodes"].shape[1], bundle["parts"][0]["record"].shape[1],
                                bundle["node_premiums"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.0001)
    fit = bundle["parts"][0]
    best, best_epoch, stale, history = float("inf"), epochs, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = (fit["weight"] * (model(bundle["nodes"], fit) - fit["scaled_y"]).square()).mean()
        assert torch.isfinite(loss)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.)
        optimizer.step()
        record = dict(epoch=epoch, train_objective=float(loss.detach()))
        if tune:
            score = ewg.metrics(bundle["parts"][1]["y"], ewg.predict(model, bundle, bundle["parts"][1]))["rmse"]
            record["inner_rmse"] = score
            if score < best - .01:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        history.append(record)
        if epoch == 1 or epoch % 40 == 0:
            print(kind, seed, "tune" if tune else "refit", epoch, "loss", round(record["train_objective"], 4), flush=True)
        if tune and stale >= ewg.PATIENCE:
            break
    return model, best_epoch, history


def by_history(frame, pred_cols):
    bucket = pd.cut(frame["history"], BUCKETS[0], labels=BUCKETS[1])
    rows = []
    for name in pred_cols:
        err = frame[name] - frame[TARGET]
        g = err.groupby(bucket, observed=False)
        rows.append(pd.DataFrame({"model": name, "history": g.size().index.astype(str), "n_rows": g.size().to_numpy(),
                                  "rmse": np.sqrt(g.apply(lambda e: np.mean(e ** 2))).to_numpy(),
                                  "bias": g.mean().to_numpy()}))
        rows.append(pd.DataFrame({"model": [name], "history": ["all"], "n_rows": [len(err)],
                                  "rmse": [float(np.sqrt(np.mean(err ** 2)))], "bias": [float(err.mean())]}))
    return pd.concat(rows, ignore_index=True)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--models", default="MLP_matched,GNN_uniform")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--max-epochs", type=int, default=320)
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args(argv)
    max_epochs = 3 if args.smoke else args.max_epochs
    ewg.PATIENCE = 2 if args.smoke else 30
    torch.set_num_threads(args.threads)
    out = OUT / "smoke" if args.smoke else OUT
    out.mkdir(parents=True, exist_ok=True)

    hdb = cmp.part23_block_table()
    tr, frames = gpb.part23_premium_frames(hdb)
    gb_preds, results = gpb.part23_lgbm(tr, frames)
    for r in results:
        print("REFERENCE", r, flush=True)

    rent, blocks = ewg.rent, ewg.blocks
    d = rent.RENT_APPROVAL_DATE
    parts = {"tuning": (rent[d < cmp.TUNE_START], rent[(d >= cmp.TUNE_START) & (d < cmp.HOLD_START)]),
             "holdout": (rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)])}
    history = {s: pd.concat(frames[s][:2]).set_index("row_id")["BLOCK_FT_N"] for s in gpb.SPLITS}
    parts = {s: tuple(gpb.with_premiums(f, pd.concat(frames[s][:2])) for f in pair) for s, pair in parts.items()}
    if args.smoke:
        parts = {s: (f.sample(3000, random_state=0).sort_index(), v.sample(800, random_state=0).sort_index())
                 for s, (f, v) in parts.items()}

    # Node premiums per split from that split's fit rows (log premium, log rows), standardised.
    node_premiums = {}
    for s in gpb.SPLITS:
        x = blocks[KEY].merge(frames[s][2], on=KEY, how="left", validate="m:1")[gpb.NODE_PREM] \
            .fillna({"NODE_BLOCK_PREM": 1.0, "NODE_LOG_N": 0.0}).to_numpy(float)
        x[:, 0] = np.log(x[:, 0])
        node_premiums[s] = torch.tensor((x - x.mean(0)) / x.std(0), dtype=torch.float32)

    (out / "config.json").write_text(json.dumps(dict(vars(args), max_epochs=max_epochs, patience=ewg.PATIENCE,
                                                     level="damped", premiums="row+nbr"), indent=2))
    bundles = {}
    for s, (f, v) in parts.items():
        bundles[s] = cmp.prepare(f, [v], "damped", list(ewg.BLOCK_NUM), gpb.PREM)
        bundles[s]["node_premiums"] = node_premiums[s]

    y = {s: parts[s][1].set_index("row_id")[TARGET] for s in gpb.SPLITS}
    gb = {s: gb_preds[("part23_E1_LightGBM", s)].reindex(y[s].index) for s in gpb.SPLITS}
    hold = pd.DataFrame({TARGET: y["holdout"], "history": history["holdout"].reindex(y["holdout"].index),
                         "E1 LightGBM": gb["holdout"]})
    gates = []
    for kind in args.models.split(","):
        tags = dict(model=kind, premiums="row+nbr", seed=args.seed)
        start = time.monotonic()
        _, epochs, _ = train(kind, bundles["tuning"], args.seed, max_epochs, tune=True)
        nn_pred = {}
        for s in gpb.SPLITS:
            fitted, _, _ = train(kind, bundles[s], args.seed, epochs)
            nn_pred[s] = pd.Series(ewg.predict(fitted, bundles[s], bundles[s]["parts"][1]), index=y[s].index)
        g = fitted.last_premium_gate.mean(1).numpy()  # holdout model, per block, eval mode
        gates.append(blocks[KEY].assign(model=kind, premium_gate=g, fit_rows=np.expm1(
            blocks[KEY].merge(frames["holdout"][2], on=KEY, how="left")["NODE_LOG_N"].fillna(0).to_numpy())))
        w = gpb.best_blend(y["tuning"], nn_pred["tuning"], gb["tuning"])
        for s in gpb.SPLITS:
            results.append(dict(split=s, **tags, epochs=epochs, n_rows=len(y[s]), **ewg.metrics(y[s], nn_pred[s])))
            results.append(dict(split=s, model=f"{kind} + E1 blend", premiums="row+nbr", seed=args.seed, weight=w,
                                n_rows=len(y[s]), **ewg.metrics(y[s], w * nn_pred[s] + (1 - w) * gb[s])))
        hold[f"{kind} row+nbr"] = nn_pred["holdout"]
        print("RESULT", tags, "epochs", epochs, "holdout", round(results[-2]["rmse"], 2),
              f"| blend w={w}", "holdout", round(results[-1]["rmse"], 2),
              "| mean premium gate", round(float(g.mean()), 3),
              "minutes", round((time.monotonic() - start) / 60, 1), flush=True)
        pd.DataFrame(results).to_csv(out / "results.csv", index=False)
    pd.concat(gates, ignore_index=True).to_csv(out / "premium_gates.csv", index=False)

    # `row` arms from gnn_premiums_blend, same seed and windows.
    prev = ROOT / "outputs" / "gnn_premiums_blend" / ("smoke" if args.smoke else "") / "predictions.csv"
    if prev.exists():
        p = pd.read_csv(prev)
        p = p[(p.split == "holdout") & (p.premiums == "row")]
        for kind, g in p.groupby("model"):
            hold[f"{kind} row"] = g.set_index("row_id")["neural"].reindex(hold.index)
    hold.reset_index().to_csv(out / "holdout_predictions.csv", index=False)
    cols = [c for c in hold.columns if c not in (TARGET, "history") and hold[c].notna().all()]
    breakdown = by_history(hold, cols)
    breakdown.round(2).to_csv(out / "by_history.csv", index=False)

    r = pd.DataFrame(results)
    arms = r[["model", "premiums"]].drop_duplicates()
    summary = r.set_index(["model", "premiums", "split"])["rmse"].unstack("split") \
        .reindex(pd.MultiIndex.from_frame(arms))[["tuning", "holdout"]]
    summary["weight"] = r[r.split == "holdout"].set_index(["model", "premiums"]).get("weight")
    summary.round(2).to_csv(out / "summary.csv")
    print("\nRMSE (SGD), Part 2.3 windows\n", summary.round(2).to_string(), flush=True)
    table = breakdown.pivot(index="model", columns="history", values="rmse").reindex(cols)[BUCKETS[1] + ["all"]]
    sizes = breakdown.drop_duplicates("history").set_index("history")["n_rows"].reindex(BUCKETS[1] + ["all"])
    print("\nHoldout RMSE by fit rows in the same block + flat type\n", table.round(1).to_string(),
          "\nrows", sizes.to_dict(), flush=True)

In [ ]:
%%module gnn_graph_tests
"""Tests of whether the graph matters: unseen blocks, shuffled edges, a planted-effect sweep, spatial correlation.

    uv run python scripts/gnn_graph_tests.py unseen --seed 0     # also: shuffled, planted (seeds 0, 1, 2)
    uv run python scripts/gnn_graph_tests.py spatial             # no training
    uv run python scripts/gnn_graph_tests.py summary             # tables from the runs above
    add --smoke for 3 epochs on a small sample

Writes outputs/gnn_graph_tests/.

unseen    A random 20% of blocks lose all their rentals from the fit rows, so they have no history. Early
          stopping uses those blocks' rows in the tuning window; the score is their rows in the holdout window.
          Everything is on the `blend` level. LightGBM E1 is refit on the same rows.
          Arms: row (own premiums), row+nbr (plus neighbours' premiums, gnn_neighbour_premiums.py), each for
          MLP_matched and GNN_uniform; GNN row and MLP row+nbr also on shuffled edges.
shuffled  GNN_uniform with `row` premiums on the `blend` level and shuffled edges, on all rows, against the real
          edges in outputs/gnn_premiums_blend/level-blend_same_seed*/.
          Shuffling relabels the blocks at random, so each keeps its number of neighbours and the graph keeps its
          shape, but neighbours are no longer nearby.
planted   Part 3.2's planted effect at several strengths, damped level, no premiums, its epoch counts. A new
          random u per seed. Block premiums are left out because they would absorb a planted block effect.
spatial   Correlation of two blocks' fit-row premiums by distance, against shuffled premiums, raw and within town.
          Blocks with at least 10 fit rows.

Decision rules, fixed before running. Gaps are paired within a seed; "clears" means mean gap > 2 x sd of the gap
over the three seeds. Three seeds make this a rough screen, not a formal test.
    R1 Neighbours' premiums help unseen blocks: row minus row+nbr clears, for that model.
    R2 Message passing helps unseen blocks: MLP minus GNN clears, for row and for row+nbr.
    R3 The real neighbours matter on unseen blocks: shuffled minus real clears, for GNN row and MLP row+nbr.
    R4 The real neighbours matter on all rows: shuffled minus real clears, for GNN row.
    R5 Smallest detectable effect: smallest planted strength where (MLP - GNN) minus the same gap at strength 0 clears.
"""
import argparse
import sys
import time
from pathlib import Path


from hdb_rentals import gnn_neighbour_premiums as gnp  # (imports torch before LightGBM)
import numpy as np
import pandas as pd
import torch
from sklearn.neighbors import BallTree

gpb, cmp, ewg = gnp.gpb, gnp.cmp, gnp.ewg
OUT = ROOT / "outputs" / "gnn_graph_tests"
TARGET, KEY, PREM, NODE_PREM = cmp.TARGET, cmp.KEY, gpb.PREM, gpb.NODE_PREM
REAL_GRAPH = ewg.GRAPH
HELD_FRACTION = 0.2
LEVEL = "blend"
BETAS = [0.0, 25.0, 50.0, 75.0, 100.0, 150.0]  # 300 pushed some rents below zero on seed 1
EARTH_KM = 6371.0088
UNSEEN_ARMS = [("MLP_matched", "row", "real"), ("GNN_uniform", "row", "real"), ("GNN_uniform", "row", "shuffled"),
               ("MLP_matched", "row+nbr", "real"), ("GNN_uniform", "row+nbr", "real"), ("MLP_matched", "row+nbr", "shuffled")]


def shuffled_graph(seed):
    """The same graph with blocks relabelled at random: same degrees and shape, neighbours no longer nearby."""
    perm = torch.as_tensor(np.random.default_rng(1000 + seed).permutation(REAL_GRAPH.n_nodes), dtype=torch.long)
    fields = {f: getattr(REAL_GRAPH, f) for f in REAL_GRAPH.__dataclass_fields__}
    fields["edge_index"] = perm[REAL_GRAPH.edge_index]
    return type(REAL_GRAPH)(**fields)


def held_frames(tr, held_bid):
    """gnn_premiums_blend.part23_premium_frames with the held blocks' rentals removed from the fit rows."""
    kept = tr[~tr["BID"].isin(held_bid)].copy()
    blocks = tr[KEY + ["BID"]].drop_duplicates("BID").sort_values("BID")
    nb, nft = int(tr["BID"].max()) + 1, int(tr["FT"].max()) + 1
    kept["REL"] = kept[TARGET] / kept.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median")

    def encode(src, dst):
        bid, g, rel = src["BID"].to_numpy(), src["G"].to_numpy(), src["REL"].to_numpy()
        sb, cb = np.bincount(bid, rel, nb), np.bincount(bid, minlength=nb).astype(float)
        sg, cg = np.bincount(g, rel, nb * nft), np.bincount(g, minlength=nb * nft).astype(float)
        db, dg = dst["BID"].to_numpy(), dst["G"].to_numpy()
        pb = (gpb.M_BLOCK + sb[db]) / (gpb.M_BLOCK + cb[db])
        pg = (gpb.M_FT * pb + sg[dg]) / (gpb.M_FT + cg[dg])
        return np.column_stack([pb, pg, cg[dg]])

    frames = {}
    for split, (start, end) in gpb.SPLITS.items():
        fit_df = kept[kept["RENT_APPROVAL_DATE"] < start].reset_index(drop=True)
        d = tr["RENT_APPROVAL_DATE"]
        val_df = tr[(d >= start) & (d < end)].reset_index(drop=True)
        own = np.empty((len(fit_df), 3))
        for f_idx, p_idx in gpb.KFold(5, shuffle=True, random_state=0).split(fit_df):
            own[p_idx] = encode(fit_df.iloc[f_idx], fit_df.iloc[p_idx])
        fit_df[PREM], val_df[PREM] = own, encode(fit_df, val_df)
        sb = np.bincount(fit_df["BID"], fit_df["REL"], nb)
        cb = np.bincount(fit_df["BID"], minlength=nb).astype(float)
        node = blocks.assign(NODE_BLOCK_PREM=(gpb.M_BLOCK + sb[blocks["BID"]]) / (gpb.M_BLOCK + cb[blocks["BID"]]),
                             NODE_LOG_N=np.log1p(cb[blocks["BID"]]))[KEY + NODE_PREM]
        frames[split] = (fit_df, val_df, node)
    return frames


def node_premiums(frame):
    x = ewg.blocks[KEY].merge(frame, on=KEY, how="left", validate="m:1")[NODE_PREM] \
        .fillna({"NODE_BLOCK_PREM": 1.0, "NODE_LOG_N": 0.0}).to_numpy(float)
    x[:, 0] = np.log(x[:, 0])
    return torch.tensor((x - x.mean(0)) / x.std(0), dtype=torch.float32)


def set_up(smoke):
    ewg.PATIENCE = 2 if smoke else 30
    return 3 if smoke else 320


def run_unseen(args, out):
    max_epochs = set_up(args.smoke)
    tr, _ = gpb.part23_premium_frames(cmp.part23_block_table())
    rng = np.random.default_rng(100 + args.seed)
    all_bid = tr["BID"].unique()
    held_bid = rng.choice(all_bid, int(HELD_FRACTION * len(all_bid)), replace=False)
    frames = held_frames(tr, held_bid)
    assert not any(f["BID"].isin(held_bid).any() for f, _, _ in frames.values())
    gb_preds, _ = gpb.part23_lgbm(tr, frames, LEVEL)
    held_rows = set(tr.loc[tr["BID"].isin(held_bid), "row_id"])
    rent, d = ewg.rent, ewg.rent.RENT_APPROVAL_DATE
    parts = {"tuning": (rent[d < cmp.TUNE_START], rent[(d >= cmp.TUNE_START) & (d < cmp.HOLD_START)]),
             "holdout": (rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)])}
    bundles = {}
    for s, (f, v) in parts.items():
        f = gpb.with_premiums(f[f.row_id.isin(frames[s][0].row_id)], frames[s][0])
        v = gpb.with_premiums(v, frames[s][1])
        if s == "tuning":
            v = v[v.row_id.isin(held_rows)]
        if args.smoke:
            f, v = f.sample(3000, random_state=0).sort_index(), v.sample(800, random_state=0).sort_index()
        bundles[s] = cmp.prepare(f, [v], LEVEL, list(ewg.BLOCK_NUM), PREM)
        bundles[s]["node_premiums"] = node_premiums(frames[s][2])
    val = bundles["holdout"]["parts"][1]["frame"]
    y, is_held = val[TARGET].to_numpy(float), val["row_id"].isin(held_rows).to_numpy()
    rows = []

    def record(name, premiums, graph, pred, epochs=np.nan):
        for slice_name, mask in [("unseen blocks", is_held), ("seen blocks", ~is_held)]:
            rows.append(dict(seed=args.seed, model=name, premiums=premiums, graph=graph, slice=slice_name,
                             epochs=epochs, n_rows=int(mask.sum()), **ewg.metrics(y[mask], pred[mask])))
        pd.DataFrame(rows).to_csv(out / f"unseen_seed{args.seed}.csv", index=False)

    record("E1 LightGBM", "row", "none", gb_preds[("part23_E1_LightGBM", "holdout")].reindex(val["row_id"]).to_numpy())
    for kind, premiums, graph in UNSEEN_ARMS:
        start = time.monotonic()
        ewg.GRAPH = REAL_GRAPH if graph == "real" else shuffled_graph(args.seed)
        trainer = ewg.train if premiums == "row" else gnp.train
        _, epochs, _ = trainer(kind, bundles["tuning"], args.seed, max_epochs, tune=True)
        fitted, _, _ = trainer(kind, bundles["holdout"], args.seed, epochs)
        record(kind, premiums, graph, ewg.predict(fitted, bundles["holdout"], bundles["holdout"]["parts"][1]), epochs)
        print("RESULT", kind, premiums, graph, "seed", args.seed, "epochs", epochs, rows[-2]["rmse"],
              "minutes", round((time.monotonic() - start) / 60, 1), flush=True)
    ewg.GRAPH = REAL_GRAPH


def run_shuffled(args, out):
    max_epochs = set_up(args.smoke)
    _, frames = gpb.part23_premium_frames(cmp.part23_block_table())
    rent, d = ewg.rent, ewg.rent.RENT_APPROVAL_DATE
    parts = {"tuning": (rent[d < cmp.TUNE_START], rent[(d >= cmp.TUNE_START) & (d < cmp.HOLD_START)]),
             "holdout": (rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)])}
    bundles = {}
    for s, pair in parts.items():
        f, v = (gpb.with_premiums(x, pd.concat(frames[s][:2])) for x in pair)
        if args.smoke:
            f, v = f.sample(3000, random_state=0).sort_index(), v.sample(800, random_state=0).sort_index()
        bundles[s] = cmp.prepare(f, [v], LEVEL, list(ewg.BLOCK_NUM), PREM)
    ewg.GRAPH = shuffled_graph(args.seed)
    _, epochs, _ = ewg.train("GNN_uniform", bundles["tuning"], args.seed, max_epochs, tune=True)
    fitted, _, _ = ewg.train("GNN_uniform", bundles["holdout"], args.seed, epochs)
    ewg.GRAPH = REAL_GRAPH
    part = bundles["holdout"]["parts"][1]
    row = dict(seed=args.seed, model="GNN_uniform", premiums="row", graph="shuffled", epochs=epochs,
               **ewg.metrics(part["y"], ewg.predict(fitted, bundles["holdout"], part)))
    pd.DataFrame([row]).to_csv(out / f"shuffled_seed{args.seed}.csv", index=False)
    print("RESULT", row, flush=True)


def run_planted(args, out):
    set_up(args.smoke)
    rent, d = ewg.rent, ewg.rent.RENT_APPROVAL_DATE
    fit, hold = rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)]
    if args.smoke:
        fit, hold = fit.sample(3000, random_state=0).sort_index(), hold.sample(800, random_state=0).sort_index()
    r = pd.read_csv(ROOT / "outputs/gnn_vs_lightgbm/results.csv")
    r = r[(r.level == "damped") & (r.features == "gnn") & (r.split == "holdout")].set_index("model")["epochs"]
    # Same draw as Part 3.2 for seed 0.
    u = np.random.default_rng(args.seed + 1).standard_normal(REAL_GRAPH.n_nodes)
    row_ids, col_ids = REAL_GRAPH.edge_index.numpy()
    degree = np.bincount(row_ids, minlength=REAL_GRAPH.n_nodes)
    total = np.bincount(row_ids, u[col_ids], minlength=REAL_GRAPH.n_nodes)
    neighbour_mean_u = np.divide(total, degree, out=np.zeros_like(u), where=degree > 0)
    ewg.blocks["PLANTED_U"] = u
    block_num = list(ewg.BLOCK_NUM) + ["PLANTED_U"]
    adjustment = cmp.MarketRentAdjustment("damped").fit(fit)

    def plant(frame, beta):
        out_frame = frame.copy()
        out_frame["MONTHLY_RENT"] = frame.MONTHLY_RENT + beta * neighbour_mean_u[frame.block_id.to_numpy()] * adjustment.multipliers(frame.MONTH_INDEX)
        assert (out_frame.MONTHLY_RENT > 0).all()
        return out_frame

    rows = []
    for beta in BETAS:
        start = time.monotonic()
        fit_p, hold_p = plant(fit, beta), plant(hold, beta)
        bundle = cmp.prepare(fit_p, [hold_p], "damped", block_num)
        effect_sd = float((hold_p.MONTHLY_RENT - hold.MONTHLY_RENT).std())
        for kind in ["MLP_matched", "GNN_uniform"]:
            epochs = 3 if args.smoke else int(r[kind])
            fitted, _, _ = ewg.train(kind, bundle, args.seed, epochs)
            part = bundle["parts"][1]
            rows.append(dict(seed=args.seed, beta=beta, effect_sd=effect_sd, model=kind, epochs=epochs,
                             **ewg.metrics(part["y"], ewg.predict(fitted, bundle, part))))
        pd.DataFrame(rows).to_csv(out / f"planted_seed{args.seed}.csv", index=False)
        print("RESULT beta", beta, "seed", args.seed, [round(x["rmse"], 2) for x in rows[-2:]],
              "minutes", round((time.monotonic() - start) / 60, 1), flush=True)


def run_spatial(args, out):
    _, frames = gpb.part23_premium_frames(cmp.part23_block_table())
    fit = frames["holdout"][0]
    g = fit.groupby(KEY).agg(prem=("REL", "mean"), n=("REL", "size")).reset_index()
    g = g[g.n >= 10].merge(ewg.blocks[KEY + ["LATITUDE", "LONGITUDE"]], on=KEY, how="inner", validate="1:1")
    g["within_town"] = g.prem - g.groupby("TOWN").prem.transform("mean")
    bands = [0, .1, .2, .3, .5, 1, 2, 5, 10]
    tree = BallTree(np.radians(g[["LATITUDE", "LONGITUDE"]].to_numpy()), metric="haversine")
    ids, dist = tree.query_radius(np.radians(g[["LATITUDE", "LONGITUDE"]].to_numpy()), r=bands[-1] / EARTH_KM, return_distance=True)
    a = np.concatenate([np.full(len(i), k) for k, i in enumerate(ids)])
    b, km = np.concatenate(ids), np.concatenate(dist) * EARTH_KM
    keep = a < b
    a, b, km = a[keep], b[keep], km[keep]
    rng = np.random.default_rng(0)
    rows = []
    for lo, hi in zip(bands[:-1], bands[1:]):
        m = (km > lo) & (km <= hi)
        row = dict(band_km=f"{lo}-{hi}", n_pairs=int(m.sum()))
        for col in ["prem", "within_town"]:
            x = g[col].to_numpy()
            row[f"corr {col}"] = float(np.corrcoef(x[a[m]], x[b[m]])[0, 1])
            null = [np.corrcoef(p[a[m]], p[b[m]])[0, 1] for p in (rng.permutation(x) for _ in range(50))]
            row[f"shuffled sd {col}"] = float(np.std(null))
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / "spatial.csv", index=False)
    print(f"{len(g)} blocks with at least 10 fit rows\n", table.round(3).to_string(), flush=True)


def clears(gaps):
    gaps = np.asarray(gaps, float)
    return dict(gap=float(gaps.mean()), sd=float(gaps.std(ddof=1)), threshold=float(2 * gaps.std(ddof=1)),
                clears=bool(gaps.mean() > 2 * gaps.std(ddof=1)))


def summarise(out):
    u = pd.concat([pd.read_csv(p) for p in sorted(out.glob("unseen_seed*.csv"))])
    table = u.groupby(["slice", "model", "premiums", "graph"], sort=False).agg(
        rmse=("rmse", "mean"), sd=("rmse", "std"), bias=("bias", "mean"), n_rows=("n_rows", "mean"), epochs=("epochs", "mean"))
    table.round(2).to_csv(out / "unseen_summary.csv")
    held = u[u.slice == "unseen blocks"].set_index(["model", "premiums", "graph", "seed"])["rmse"].unstack("seed")
    arm = lambda m, p, g="real": held.loc[(m, p, g)].to_numpy()
    rules = [("R1 neighbours' premiums help, MLP", arm("MLP_matched", "row") - arm("MLP_matched", "row+nbr")),
             ("R1 neighbours' premiums help, GNN", arm("GNN_uniform", "row") - arm("GNN_uniform", "row+nbr")),
             ("R2 message passing helps, row", arm("MLP_matched", "row") - arm("GNN_uniform", "row")),
             ("R2 message passing helps, row+nbr", arm("MLP_matched", "row+nbr") - arm("GNN_uniform", "row+nbr")),
             ("R3 real neighbours matter, GNN row", arm("GNN_uniform", "row", "shuffled") - arm("GNN_uniform", "row")),
             ("R3 real neighbours matter, MLP row+nbr", arm("MLP_matched", "row+nbr", "shuffled") - arm("MLP_matched", "row+nbr"))]
    s = pd.concat([pd.read_csv(p) for p in sorted(out.glob("shuffled_seed*.csv"))]).set_index("seed").sort_index()
    real = pd.concat([pd.read_csv(ROOT / f"outputs/gnn_premiums_blend/level-{LEVEL}_same_seed{k}/results.csv").assign(seed=k) for k in s.index])
    real = real[(real.split == "holdout") & (real.model == "GNN_uniform")].set_index("seed").sort_index()
    rules.append(("R4 real neighbours matter on all rows, GNN row", s["rmse"].to_numpy() - real["rmse"].to_numpy()))
    rule_table = pd.DataFrame([dict(rule=name, **clears(g)) for name, g in rules])
    rule_table.round(2).to_csv(out / "rules.csv", index=False)
    pd.DataFrame({"graph": ["real", "shuffled"], "rmse": [real.rmse.mean(), s.rmse.mean()], "sd": [real.rmse.std(), s.rmse.std()],
                  "bias": [real.bias.mean(), s.bias.mean()]}).round(2).to_csv(out / "shuffled_summary.csv", index=False)
    p = pd.concat([pd.read_csv(f) for f in sorted(out.glob("planted_seed*.csv"))])
    p = p[p.beta.isin(BETAS)]
    wide = p.pivot(index=["seed", "beta"], columns="model", values="rmse")
    wide["gap"] = wide["MLP_matched"] - wide["GNN_uniform"]
    gap = wide["gap"].unstack("beta")
    rows = []
    for beta in gap.columns:
        beyond = clears((gap[beta] - gap[0.0]).to_numpy()) if beta else dict(gap=0.0, sd=0.0, threshold=0.0, clears=False)
        sub = wide.xs(beta, level="beta")
        rows.append(dict(beta=beta, effect_sd=p[p.beta == beta].effect_sd.mean(), MLP=sub["MLP_matched"].mean(), GNN=sub["GNN_uniform"].mean(),
                         gap=gap[beta].mean(), gap_vs_zero=beyond["gap"], threshold=beyond["threshold"], clears=beyond["clears"]))
    sweep = pd.DataFrame(rows)
    sweep.round(2).to_csv(out / "planted_summary.csv", index=False)
    print(table.round(2).to_string(), "\n\n", rule_table.round(2).to_string(), "\n\n", sweep.round(2).to_string(), flush=True)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("experiment", choices=["unseen", "shuffled", "planted", "spatial", "summary"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--threads", type=int, default=4)
    ap.add_argument("--level", default="blend", choices=["blend", "hold"], help="level for unseen and shuffled; other than blend writes to a subfolder")
    args = ap.parse_args(argv)
    torch.set_num_threads(args.threads)
    global LEVEL
    LEVEL = args.level
    base = OUT if LEVEL == "blend" else OUT / f"level-{LEVEL}"
    out = base / "smoke" if args.smoke else base
    out.mkdir(parents=True, exist_ok=True)
    {"unseen": run_unseen, "shuffled": run_shuffled, "planted": run_planted, "spatial": run_spatial,
     "summary": lambda a, o: summarise(o)}[args.experiment](args, out)

In [ ]:
%%module explain_table2_gnn
"""Explain the model the report scores: GNN_uniform with `row` premiums on the `blend` level.

    main()  # or main(level="hold")

Same frames, inputs, level and epoch counts as `gnn_premiums_blend.main(["--premiums", "row", "--level", "blend",
"--same-level"])` (seeds 0, 1, 2), with Part 3.2's explainer and checks. The own-block premiums are row
inputs here, so IG attributes them with the flat's record, and neighbours carry only block features.
The planted-effect check is not repeated: a block's premium would absorb a planted neighbour effect
(Part 2.4), so it stays a check of the method on the premium-free GNN (Part 3.2).

Writes outputs/13_interpretability_table2/: models/, accuracy.csv, importance.csv, reliance.csv,
substitution.csv, reach.csv, checks.csv, fidelity.csv, stability.csv, neighbour_checks.csv,
by_history.csv and two figures.
"""
from pathlib import Path


from hdb_rentals import gnn_premiums_blend as pb  # (imports torch before LightGBM)
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hdb_rentals import gnn
from hdb_rentals.explain import SubgraphExplainer
from scipy.stats import spearmanr

cmp, ewg = pb.cmp, pb.ewg
LEVEL = "blend"  # main(level="hold") switches LEVEL, OUT and RESULTS
OUT = ROOT / "outputs" / "13_interpretability_table2"
SEEDS, N_ROWS, N_PER_BUCKET, IG_STEPS = [0, 1, 2], 400, 120, 32
RESULTS = ROOT / "outputs" / "gnn_premiums_blend" / f"level-{LEVEL}_same_seed{{}}" / "results.csv"
BLOCK_CAT = ewg.BLOCK_CAT + ["SUBZONE"]
RECORD_NUM, RECORD_CAT = ["FLOOR_AREA_SQM", "FLAT_AGE", *pb.PREM], ["FLAT_TYPE", "FLAT_MODEL"]
FLAT_FAMS = ["Floor area", "Flat type", "Flat model", "Flat age"]
FAMS = [*FLAT_FAMS, "Block premiums", "Town and area", "Location and building", "Neighbouring blocks"]
BUCKETS = ["0", "1-2", "3-5", "6-10", ">10"]
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e2e1dc", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlelocation": "left",
    "axes.titlesize": 11, "font.size": 9.5, "axes.axisbelow": True,
})


def source_column(name, cat):
    kind, rest = name.split("__", 1)
    if kind == "num":
        return rest.removeprefix("missingindicator_")
    return next(c for c in cat if rest.startswith(c + "_"))


def family(col):
    named = {"FLOOR_AREA_SQM": "Floor area", "FLAT_AGE": "Flat age", "LEASE_COMMENCE_DATE": "Flat age",
             "FLAT_TYPE": "Flat type", "FLAT_MODEL": "Flat model", "TOWN": "Town and area",
             "PLANNING_AREA": "Town and area", "SUBZONE": "Town and area"}
    if col in pb.PREM:
        return "Block premiums"
    return named.get(col, "Location and building")


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def main(level="blend"):
    global LEVEL, OUT, RESULTS
    LEVEL = level
    OUT = ROOT / "outputs" / "13_interpretability_table2" / ("" if level == "blend" else f"level-{level}")
    RESULTS = ROOT / "outputs" / "gnn_premiums_blend" / f"level-{level}_same_seed{{}}" / "results.csv"
    (OUT / "models").mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(4)
    ewg.PATIENCE = 30

    # The same data flow as gnn_premiums_blend.main, for the holdout split.
    hdb = cmp.part23_block_table()
    tr, frames = pb.part23_premium_frames(hdb)
    rent, blocks, graph = ewg.rent, ewg.blocks, ewg.GRAPH
    d = rent.RENT_APPROVAL_DATE
    fit_rent, hold_rent = rent[d < cmp.HOLD_START], rent[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)]
    fit_lgb, hold_lgb, _ = frames["holdout"]
    all_prem = pd.concat([fit_lgb, hold_lgb])
    fit, hold = pb.with_premiums(fit_rent, all_prem), pb.with_premiums(hold_rent, all_prem)
    bundle = cmp.prepare(fit, [hold], LEVEL, list(ewg.BLOCK_NUM), pb.PREM)
    part, nodes = bundle["parts"][1], bundle["nodes"]
    dollars = bundle["std"] * part["scale"]

    # Column names, by re-fitting the preprocessors; the asserts check they match prepare's matrices.
    bp = ewg.preprocessor(list(ewg.BLOCK_NUM), BLOCK_CAT).fit(blocks.iloc[np.sort(fit.block_id.unique())])
    rp = ewg.preprocessor(RECORD_NUM, RECORD_CAT).fit(fit)
    assert np.allclose(bp.transform(blocks), nodes.numpy(), atol=1e-5)
    assert np.allclose(rp.transform(hold), part["record"].numpy(), atol=1e-5)
    node_fam = np.array([family(source_column(n, BLOCK_CAT)) for n in bp.get_feature_names_out()])
    rec_names = [source_column(n, RECORD_CAT) for n in rp.get_feature_names_out()]
    rec_fam = np.array([family(c) for c in rec_names])
    premium_cols = np.flatnonzero(rec_fam == "Block premiums")  # reset together: they are one signal
    count_col = rec_names.index("BLOCK_FT_N")
    level_cols = [c for c in premium_cols if c != count_col]

    # Models, on the epoch counts of the Table II runs.
    recorded = {s: pd.read_csv(str(RESULTS).format(s)).query("split == 'holdout' and premiums == 'row'").set_index("model")
                for s in SEEDS}

    def get_model(kind, seed):
        path = OUT / "models" / f"{kind}_seed{seed}.pt"
        if path.exists():
            return gnn.load_model(path, graph)[0]
        model, _, _ = ewg.train(kind, bundle, seed, int(recorded[seed].epochs[kind]), save_to=path)
        return model.eval()

    models = {f"GNN seed {s}": get_model("GNN_uniform", s) for s in SEEDS}
    models["MLP"] = get_model("MLP_matched", 0)
    accuracy = [{"model": name, "holdout rmse": rmse(gnn.predict(m, bundle, part), part["y"]),
                 "table ii run": recorded[0 if name == "MLP" else int(name[-1])].rmse["MLP_matched" if name == "MLP" else "GNN_uniform"]}
                for name, m in models.items()]
    gnn.seed_everything(99)
    models["GNN untrained"] = gnn.WeightedRentalGNN("GNN_uniform", nodes.shape[1], part["record"].shape[1], graph).eval()

    # LightGBM with premiums on the same level, as in Table II, for TreeSHAP.
    num, cat = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE"] + cmp.DIST, ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
    categories = {c: pd.CategoricalDtype(sorted(tr[c].unique())) for c in cat}

    def matrix(df):
        X = df[num + pb.PREM + cat].copy()
        for c in cat:
            X[c] = X[c].astype(categories[c])
        return X

    adj = cmp.MarketRentAdjustment(LEVEL).fit(fit_rent)
    fs, vs = adj.multipliers(fit_lgb["MONTH_INDEX"]), adj.multipliers(hold_lgb["MONTH_INDEX"])
    est = lgb.LGBMRegressor(**cmp.LGBM_TUNED, verbose=-1, n_jobs=-1, random_state=0)
    est.fit(matrix(fit_lgb), fit_lgb[cmp.TARGET].to_numpy() / fs, sample_weight=fs ** 2)
    pred = est.predict(matrix(hold_lgb)) * vs
    accuracy.append({"model": "LightGBM", "holdout rmse": rmse(pred, hold_lgb[cmp.TARGET].to_numpy()),
                     "table ii run": recorded[0].rmse["part23_E1_LightGBM"]})
    pd.DataFrame(accuracy).round(2).to_csv(OUT / "accuracy.csv", index=False)
    print(pd.DataFrame(accuracy).round(2).to_string(), flush=True)

    # Rows: a random sample for overall results, and a sample per history bucket.
    explainer = SubgraphExplainer(graph.edge_index, graph.n_nodes)
    node_base, rec_base = nodes[np.sort(fit.block_id.unique())].mean(0), bundle["parts"][0]["record"].mean(0)
    rng = np.random.default_rng(0)
    rows = np.sort(rng.choice(len(hold), N_ROWS, replace=False))
    history = pd.cut(np.expm1(hold["BLOCK_FT_N"].to_numpy()).round(), [-1, 0, 2, 5, 10, np.inf], labels=BUCKETS)
    bucket_rows = {b: np.sort(rng.choice(np.flatnonzero(history == b), N_PER_BUCKET, replace=False)) for b in BUCKETS}

    @torch.no_grad()
    def full_outputs(model, edge_mask=None):
        return model(nodes, part, edge_mask).numpy()

    def explain(model, which):
        f_full = full_outputs(model)
        out = []
        for i in which:
            e = explainer.explain(model, nodes, part["record"][i], int(part["ids"][i]), row=int(i),
                                  node_base=node_base, record_base=rec_base, steps=IG_STEPS)
            out.append(dict(row=int(i), block=e.block, S=e.nodes, lb=e.local, rec=e.record, own=e.own, nbr=e.neighbours,
                            nbr_blocks=e.neighbour_blocks, f_full=float(f_full[i]), f_sub=e.output, base=e.baseline))
        return out

    def history_parts(e):
        d_i, rec, own = dollars[e["row"]], e["rec"], pd.Series(e["own"]).groupby(node_fam).sum()
        return {"own premiums": abs(rec[level_cols].sum()) * d_i, "own row count": abs(rec[count_col]) * d_i,
                "own block features": abs(own[["Town and area", "Location and building"]].sum()) * d_i,
                "neighbours": abs(e["nbr"].sum()) * d_i,
                "flat": abs(rec[[i for i in range(len(rec)) if i not in premium_cols]].sum()) * d_i}

    def families(e):
        d_i = dollars[e["row"]]
        s = pd.concat([pd.Series(e["rec"] * d_i).groupby(rec_fam).sum(), pd.Series(e["own"] * d_i).groupby(node_fam).sum()])
        s = s.groupby(level=0).sum()
        s["Neighbouring blocks"] = float(e["nbr"].sum() * d_i)
        return s.reindex(FAMS).fillna(0)

    explanations = {name: explain(m, rows) for name, m in models.items()}
    tables = {name: pd.DataFrame([families(e) for e in ex]) for name, ex in explanations.items()}
    print("explained overall rows", flush=True)

    # Which features matter, against TreeSHAP on the same rows.
    row_ids = hold["row_id"].to_numpy()[rows]
    positions = pd.Series(np.arange(len(hold_lgb)), index=hold_lgb["row_id"].to_numpy()).loc[row_ids].to_numpy()
    X = matrix(hold_lgb).iloc[positions]
    contrib = est.predict(X, pred_contrib=True)[:, :-1] * vs[positions, None]
    shap = pd.DataFrame(contrib, columns=X.columns).T.groupby(lambda c: family(c)).sum().T
    importance = pd.DataFrame({"GNN (IG)": tables["GNN seed 0"].abs().mean(), "MLP (IG)": tables["MLP"].abs().mean(),
                               "LightGBM (TreeSHAP)": shap.abs().mean().reindex(FAMS)})
    importance.round(1).to_csv(OUT / "importance.csv")
    both = importance.drop("Neighbouring blocks")
    print(importance.round(1).to_string(), "\nGNN vs LightGBM Spearman", spearmanr(both["GNN (IG)"], both["LightGBM (TreeSHAP)"]).statistic, flush=True)

    # Does the GNN use its neighbours?
    reliance = []
    for name in ["GNN seed 0", "GNN seed 1", "GNN seed 2", "MLP"]:
        m, ex = models[name], explanations[name]
        nbr = np.array([np.abs(e["nbr"]).sum() * dollars[e["row"]] for e in ex])
        tot = np.array([(np.abs(e["rec"]).sum() + np.abs(e["own"]).sum() + np.abs(e["nbr"]).sum()) * dollars[e["row"]] for e in ex])
        moved = np.abs(full_outputs(m) - full_outputs(m, torch.zeros(graph.n_edges))) * dollars
        t = tables[name].abs().mean()
        reliance.append({"model": name, "neighbour share": nbr.sum() / tot.sum(),
                         "neighbour share of net effects": t["Neighbouring blocks"] / t.sum(),
                         "mean moved with no edges": moved.mean()})
    pd.DataFrame(reliance).round(3).to_csv(OUT / "reliance.csv", index=False)

    def location_effect(t, parts):
        return t[parts].sum(axis=1).abs().mean()

    g, m = tables["GNN seed 0"], tables["MLP"]
    block_fams = ["Block premiums", "Town and area", "Location and building"]
    pd.Series({"GNN: own block, premiums included": location_effect(g, block_fams),
               "GNN: neighbours": g["Neighbouring blocks"].abs().mean(),
               "GNN: own block + neighbours": location_effect(g, [*block_fams, "Neighbouring blocks"]),
               "MLP: own block, premiums included": location_effect(m, block_fams)}).round(1).to_csv(OUT / "substitution.csv")

    records = []
    for e in explanations["GNN seed 0"]:
        hop = explainer.hop(e["block"], e["nbr_blocks"])
        records.extend({"hop": int(h), "dollars": abs(a) * dollars[e["row"]]} for h, a in zip(hop, e["nbr"].sum(1)))
    pd.DataFrame(records).groupby("hop").dollars.agg(["size", "mean"]).round(2).to_csv(OUT / "reach.csv")

    # Checks: exactness, completeness, seed stability, randomisation, fidelity.
    checks = []
    for name, ex in explanations.items():
        comp = [abs(e["rec"].sum() + e["own"].sum() + e["nbr"].sum() - (e["f_sub"] - e["base"])) * dollars[e["row"]] for e in ex]
        checks.append({"model": name, "max subgraph vs full": max(abs(e["f_sub"] - e["f_full"]) for e in ex),
                       "median completeness error $": np.median(comp), "max completeness error $": np.max(comp)})
    pd.DataFrame(checks).to_csv(OUT / "checks.csv", index=False)

    def row_spearman(a, b):
        return float(np.nanmean([spearmanr(x, y).statistic for x, y in zip(tables[a].abs().to_numpy(), tables[b].abs().to_numpy())]))

    pairs = [("GNN seed 0", "GNN seed 1"), ("GNN seed 0", "GNN seed 2"), ("GNN seed 1", "GNN seed 2"),
             ("GNN seed 0", "GNN untrained"), ("GNN seed 0", "MLP")]
    pd.DataFrame([{"pair": f"{a} vs {b}", "mean per-rental spearman": row_spearman(a, b)} for a, b in pairs]).round(3).to_csv(OUT / "stability.csv", index=False)

    @torch.no_grad()
    def fidelity(name, ks=(1, 3, 5, 10, 20), draws=5):
        r_ng = np.random.default_rng(0)
        model, res = models[name], {k: ([], [], []) for k in ks}
        for e in explanations[name]:
            i, S, lb = e["row"], e["S"], e["lb"]
            sub = explainer.submodel(model, S)
            x, r = nodes[S].clone(), part["record"][i].clone()
            attr = np.r_[e["rec"], np.insert(e["nbr"], lb, e["own"], axis=0).ravel()]
            diff = np.r_[(r - rec_base).numpy(), (x - node_base).numpy().ravel()]
            pool = np.flatnonzero(np.abs(diff) > 1e-9)
            n_local = len(r) + (lb + 1) * x.shape[1]
            own_pool = pool[(pool < len(r)) | ((pool >= len(r) + lb * x.shape[1]) & (pool < n_local))]
            top = pool[np.argsort(-np.abs(attr[pool]))]
            base_flat = np.r_[rec_base.numpy(), node_base.expand_as(x).numpy().ravel()]
            full_flat = np.r_[r.numpy(), x.numpy().ravel()]

            def score(idx):
                if np.isin(idx, premium_cols).any():
                    idx = np.union1d(idx, premium_cols)
                v = full_flat.copy()
                v[idx] = base_flat[idx]
                rr = torch.tensor(v[:len(r)], dtype=torch.float32)
                xx = torch.tensor(v[len(r):].reshape(x.shape), dtype=torch.float32)
                return float(sub(xx, dict(ids=torch.tensor([lb]), record=rr[None]))[0])

            for k in ks:
                if k > len(pool):
                    continue
                res[k][0].append(abs(score(top[:k]) - e["f_sub"]) * dollars[i])
                res[k][1].append(np.mean([abs(score(r_ng.choice(pool, k, replace=False)) - e["f_sub"]) * dollars[i] for _ in range(draws)]))
                if k <= len(own_pool):
                    res[k][2].append(np.mean([abs(score(r_ng.choice(own_pool, k, replace=False)) - e["f_sub"]) * dollars[i] for _ in range(draws)]))
        return pd.DataFrame({k: {"top": np.mean(t), "random whole subgraph": np.mean(rd), "random flat and own block": np.mean(ro)}
                             for k, (t, rd, ro) in res.items()}).T

    fidelity("GNN seed 0").round(1).to_csv(OUT / "fidelity.csv")

    # Do the attributions rank individual direct neighbours? Reset one neighbour's features to the
    # baseline, the most-attributed against a random one, and compare the ranking across seeds.
    @torch.no_grad()
    def neighbour_checks(name, seed=0):
        r_ng, top, rand = np.random.default_rng(seed), [], []
        model = models[name]
        for e in explanations[name]:
            direct = np.flatnonzero(explainer.hop(e["block"], e["nbr_blocks"]) == 1)
            if len(direct) < 2:
                continue
            i, S, lb = e["row"], e["S"], e["lb"]
            sub = explainer.submodel(model, S)

            def moved(k):
                x = nodes[S].clone()
                x[int(np.searchsorted(S, e["nbr_blocks"][k]))] = node_base
                return abs(float(sub(x, dict(ids=torch.tensor([lb]), record=part["record"][i][None]))[0]) - e["f_sub"]) * dollars[i]

            totals = np.abs(e["nbr"][direct].sum(1))
            top.append(moved(direct[int(np.argmax(totals))]))
            rand.append(moved(r_ng.choice(direct)))
        return {"top neighbour moved": float(np.mean(top)), "random neighbour moved": float(np.mean(rand)), "rows": len(top)}

    def neighbour_rank(a, b):
        vals = []
        for ea, eb in zip(explanations[a], explanations[b]):
            direct = np.flatnonzero(explainer.hop(ea["block"], ea["nbr_blocks"]) == 1)
            if len(direct) >= 3:
                vals.append(spearmanr(np.abs(ea["nbr"][direct].sum(1)), np.abs(eb["nbr"][direct].sum(1))).statistic)
        return float(np.nanmean(vals))

    nc = neighbour_checks("GNN seed 0")
    for a, b in pairs:
        if "MLP" not in b:
            nc[f"neighbour rank {a} vs {b}"] = neighbour_rank(a, b)
    pd.Series(nc).round(3).to_csv(OUT / "neighbour_checks.csv")
    print("checks done", flush=True)

    # By history: the rows where neighbours should matter most.
    history_rows = []
    for name in ["GNN seed 0", "MLP"]:
        for bucket, which in bucket_rows.items():
            t = pd.DataFrame([history_parts(e) for e in explain(models[name], which)]).mean()
            history_rows.append({"model": name, "history": bucket, **t.to_dict()})
    by_history = pd.DataFrame(history_rows)
    by_history.round(1).to_csv(OUT / "by_history.csv", index=False)
    print(by_history.round(1).to_string(), flush=True)

    # Figures.
    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    y = np.arange(len(FAMS))
    for j, (col, color) in enumerate(zip(["GNN (IG)", "LightGBM (TreeSHAP)"], [BLUE, ORANGE])):
        ax.barh(y + (j - 0.5) * 0.38, importance[col].fillna(0), height=0.34, color=color, label=col)
    ax.set_yticks(y, FAMS)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("mean |contribution|, SGD per month")
    ax.set_title("What moves the prediction away from an average flat")
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(OUT / "importance_table2.png", dpi=200)

    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4), sharey=True)
    parts = [("own premiums", "Own block's premiums", BLUE), ("own block features", "Own block's features", AQUA), ("neighbours", "Neighbouring blocks", ORANGE)]
    for ax, name, title in zip(axes, ["GNN seed 0", "MLP"], ["GNN", "MLP (graph off)"]):
        t = by_history[by_history.model == name].set_index("history").reindex(BUCKETS)
        x = np.arange(len(BUCKETS))
        for j, (col, label, color) in enumerate(parts):
            ax.bar(x + (j - 1) * 0.27, t[col], width=0.24, color=color, label=label)
        ax.set_xticks(x, BUCKETS)
        ax.grid(axis="x", visible=False)
        ax.set_title(title)
        ax.set_xlabel("earlier training rentals of the same block + flat type")
    axes[0].set_ylabel("mean |contribution|, SGD per month")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT / "history_table2.png", dpi=200)
    print("DONE", flush=True)

## 4. Market-level evaluation for the report

Writes `outputs/report_level/`, which notebook 3 Appendix D reads. Loads its data when imported.

In [ ]:
%%module report_level_eval
"""Every number and the figure in the report's market-level section (4b_level_eval.tex).

    uv run python scripts/report_level_eval.py

Rebuilt from the report's text, because the code behind that section is not in this
repository. Uses Part 2.3's E0 columns and tuned settings (compare_gnn_lightgbm.py).
Column order follows E0, since LightGBM's column sampling depends on it: 487.49 for the
month index needs this order. Writes outputs/report_level/:

    table1.csv           Table I, on the holdout
    macro.csv            COE, share and placebo columns added to the month-index model
    macro_checks.csv     the same over five seeds, and tuned against the older untuned settings
    no_location.csv      the same level change without location features
    rolling.csv          RMSE by forecast origin, each scoring the next 16 months
    rolling_premiums.csv the same on the submitted model's columns, with block premiums rebuilt per origin
    level_backtest.png   Fig. 1, from rolling.csv
"""
import sys
from pathlib import Path


from hdb_rentals import compare_gnn_lightgbm as cmp
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hdb_rentals.baselines import LGBM_PARAMS
from hdb_rentals.macro import load_macro
from hdb_rentals.market_level import forecast_level, hedonic_level
from sklearn.model_selection import KFold

OUT = ROOT / "outputs" / "report_level"
CAT = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
FIT_END, HOLD_START, HOLD_END = 35, 36, 50  # 2023-12, 2024-01, 2025-03
ORIGINS = range(11, 35)  # 2021-12 to 2023-11
LEVELS = ["hold", "damped", "average", "straight line", "oracle"]
METHOD = {"hold": "hold", "damped": "damped", "average": "blend", "straight line": "linear"}


def load():
    hdb = cmp.part23_block_table()
    tr = cmp.clean_rent(pd.read_csv(cmp.DATA / "train.csv")).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
    tr = tr.merge(hdb[cmp.KEY + cmp.DIST], on=cmp.KEY, how="left", validate="m:1")
    tr["FLAT_AGE"] = tr["RENT_APPROVAL_DATE"].dt.year - tr["LEASE_COMMENCE_DATE"]
    hedonic = tr[["MONTH_INDEX", cmp.TARGET, *CAT, "FLOOR_AREA_SQM", "FLAT_AGE"]].copy()
    for c in CAT:
        tr[c] = tr[c].astype(pd.CategoricalDtype(sorted(tr[c].unique())))
    macro = load_macro(gate=False)
    tr = tr.join(macro.reindex(tr["MONTH_INDEX"]).reset_index(drop=True))
    stocks = pd.read_csv(cmp.DATA / "auxiliary" / "sg-stock-prices.csv")
    penguin = stocks[stocks["NAME"].str.contains("Penguin")]
    month = pd.to_datetime(penguin["DATE"]).dt.to_period("M")
    placebo = penguin.groupby(month)["ADJUSTED_CLOSE"].mean()
    placebo.index = [(p.year - 2021) * 12 + p.month - 1 for p in placebo.index]
    tr["PLACEBO"] = placebo.reindex(tr["MONTH_INDEX"]).to_numpy()
    return tr, hedonic


TR, HEDONIC = load()
MONTHS = TR["MONTH_INDEX"].to_numpy()
Y = TR[cmp.TARGET].to_numpy(float)
E0_LEVEL = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", *cmp.DIST, *CAT]
E0 = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX", *cmp.DIST, *CAT]
NO_LOCATION = ["FLOOR_AREA_SQM", "FLAT_AGE", *CAT]


def fit_predict(cols, fit, val, params=cmp.LGBM_TUNED, scale=None, seed=0):
    """LightGBM on rent, or on rent / scale and multiplied back when scale is given."""
    s = np.ones(len(Y)) if scale is None else scale
    est = lgb.LGBMRegressor(**params, verbose=-1, n_jobs=-1, random_state=seed)
    return est.fit(TR[cols].iloc[fit], Y[fit] / s[fit]).predict(TR[cols].iloc[val]) * s[val]


def level_scale(level, fit, val):
    """Market level per row of fit and val, from fit months only unless oracle."""
    if level == "oracle":
        log_level = hedonic_level(HEDONIC, np.concatenate([fit, val]))
    else:
        ahead = np.arange(MONTHS[fit].max() + 1, MONTHS[val].max() + 1)
        log_level = forecast_level(hedonic_level(HEDONIC, fit), ahead, METHOD[level])
    scale = np.full(len(Y), np.nan)
    rows = np.concatenate([fit, val])
    scale[rows] = np.exp(log_level.reindex(MONTHS[rows]).to_numpy())
    return scale


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def row(name, pred, val):
    ahead, err = MONTHS[val] - FIT_END, pred - Y[val]
    near, far = ahead <= 3, ahead >= 13
    return {"arm": name, "rmse": rmse(pred, Y[val]), "bias": err.mean(),
            "bias_1_3": err[near].mean(), "bias_13_15": err[far].mean(),
            "rmse_1_3": rmse(pred[near], Y[val][near]), "rmse_13_15": rmse(pred[far], Y[val][far])}


def holdout_tables():
    fit = np.where(MONTHS <= FIT_END)[0]
    val = np.where((MONTHS >= HOLD_START) & (MONTHS <= HOLD_END))[0]

    table = [row("month index", fit_predict(E0, fit, val), val)]
    for level in LEVELS:
        table.append(row(level, fit_predict(E0_LEVEL, fit, val, scale=level_scale(level, fit, val)), val))
    pd.DataFrame(table).round(2).to_csv(OUT / "table1.csv", index=False)

    arms = {"COE_B, COE_E": ["COE_B", "COE_E"], "shares": ["STOCK_BANKS", "STOCK_PROPERTY", "STOCK_BROAD"],
            "placebo": ["PLACEBO"]}
    macro = [row(name, fit_predict(E0 + cols, fit, val), val) for name, cols in arms.items()]
    pd.DataFrame(macro).round(2).to_csv(OUT / "macro.csv", index=False)

    coe, shares = arms["COE_B, COE_E"], arms["shares"]
    checks = []
    for seed in range(5):
        for name, cols in [("month index", E0), ("+ COE", E0 + coe), ("+ shares", E0 + shares), ("+ placebo", E0 + arms["placebo"])]:
            checks.append({"check": "tuned, seed", "seed": seed, "arm": name, "rmse": rmse(fit_predict(cols, fit, val, seed=seed), Y[val])})
    for name, cols in [("month index", E0), ("+ COE", E0 + coe)]:
        checks.append({"check": "untuned settings", "seed": 0, "arm": name, "rmse": rmse(fit_predict(cols, fit, val, LGBM_PARAMS), Y[val])})
    pd.DataFrame(checks).round(2).to_csv(OUT / "macro_checks.csv", index=False)

    average = level_scale("average", fit, val)
    rows = []
    for name, params in [("tuned", cmp.LGBM_TUNED), ("default", LGBM_PARAMS)]:
        rows.append(row(f"{name}, month index", fit_predict([*NO_LOCATION, "MONTH_INDEX"], fit, val, params), val))
        rows.append(row(f"{name}, average", fit_predict(NO_LOCATION, fit, val, params, average), val))
    pd.DataFrame(rows).round(2).to_csv(OUT / "no_location.csv", index=False)


def rolling():
    rows = []
    for origin in ORIGINS:
        fit = np.where(MONTHS <= origin)[0]
        val = np.where((MONTHS > origin) & (MONTHS <= origin + 16))[0]
        r = {"origin": origin, "month index": rmse(fit_predict(E0, fit, val), Y[val])}
        for level in LEVELS:
            r[level] = rmse(fit_predict(E0_LEVEL, fit, val, scale=level_scale(level, fit, val)), Y[val])
        rows.append(r)
        pd.DataFrame(rows).round(2).to_csv(OUT / "rolling.csv", index=False)
        print(r, flush=True)


def block_premiums(fit, val):
    """Part 2.3 E1 for one origin: `oof` premiums for fit rows, all fit rows for scored rows."""
    block = (TR["TOWN"].astype(str) + "|" + TR["BLOCK"].astype(str) + "|" + TR["STREET"].astype(str)).astype("category").cat.codes.to_numpy(dtype=np.int64)
    flat = TR["FLAT_TYPE"].cat.codes.to_numpy(dtype=np.int64)
    nb, nft = block.max() + 1, flat.max() + 1
    group = block * nft + flat
    rel = Y / TR.groupby(["MONTH_INDEX", "FLAT_TYPE"], observed=True)[cmp.TARGET].transform("median").to_numpy()

    def encode(src, dst):
        sb, cb = np.bincount(block[src], rel[src], nb), np.bincount(block[src], minlength=nb)
        sg, cg = np.bincount(group[src], rel[src], nb * nft), np.bincount(group[src], minlength=nb * nft)
        pb = (20 + sb[block[dst]]) / (20 + cb[block[dst]])
        return np.column_stack([pb, (5 * pb + sg[group[dst]]) / (5 + cg[group[dst]]), cg[group[dst]]])

    out = np.full((len(Y), 3), np.nan)
    for train_idx, own_idx in KFold(5, shuffle=True, random_state=0).split(fit):
        out[fit[own_idx]] = encode(fit[train_idx], fit[own_idx])
    out[val] = encode(fit, val)
    return out


def rolling_premiums():
    """Rolling origins on the submitted model's columns. Level arms weight rows by the level squared, as Part 2.3 E6."""
    prem = ["BLOCK_PREM", "BLOCK_FT_PREM", "BLOCK_FT_N"]
    num, time_num = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE"], ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"]
    rows = []
    for origin in ORIGINS:
        fit = np.where(MONTHS <= origin)[0]
        val = np.where((MONTHS > origin) & (MONTHS <= origin + 16))[0]
        frame = TR.copy()
        frame[prem] = block_premiums(fit, val)

        def run(cols, scale=None):
            s = np.ones(len(Y)) if scale is None else scale
            est = lgb.LGBMRegressor(**cmp.LGBM_TUNED, verbose=-1, n_jobs=-1, random_state=0)
            weight = None if scale is None else s[fit] ** 2
            est.fit(frame[cols].iloc[fit], Y[fit] / s[fit], sample_weight=weight)
            return est.predict(frame[cols].iloc[val]) * s[val]

        r = {"origin": origin, "month index": rmse(run([*time_num, *cmp.DIST, *prem, *CAT]), Y[val])}
        for level in LEVELS:
            r[level] = rmse(run([*num, *cmp.DIST, *prem, *CAT], level_scale(level, fit, val)), Y[val])
        rows.append(r)
        pd.DataFrame(rows).round(2).to_csv(OUT / "rolling_premiums.csv", index=False)
        print(r, flush=True)


def figure():
    r = pd.read_csv(OUT / "rolling.csv")
    ink, ink_2, muted, grid, axis, surface = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
    styles = [("straight line", "Straight line", axis, 1.6), ("hold", "Hold", "#eb6834", 2),
              ("damped", "Damped trend", "#1baf7a", 2), ("average", "Average of hold and damped", "#2a78d6", 3.2)]
    months = pd.period_range("2021-01", periods=60, freq="M").to_timestamp()
    x = months[r["origin"]]
    fig, ax = plt.subplots(figsize=(8, 4.2), facecolor=surface)
    ax.set_facecolor(surface)
    # The seven origins from 2022-07 to 2023-01 whose scored windows span the flattening, as in the text.
    ax.axvspan(months[18] - pd.Timedelta(days=15), months[24] + pd.Timedelta(days=15), color="#eaebe6", zorder=0)
    ax.text(months[18], 335, "windows that span\nthe 2023 flattening", color=ink_2, va="top", fontsize=9.5)
    for key, label, color, width in styles:
        ax.plot(x, (r[key] - r["oracle"]).clip(lower=0), color=color, lw=width, marker="o", ms=3, label=label)
    ax.set_ylim(0, 345)
    ax.set_title("Cost of the level forecast, by starting month", loc="left", fontweight="bold", color=ink)
    ax.set_ylabel("RMSE above the oracle ($)", color=ink_2)
    ax.set_xlabel("Starting month (trained on everything up to here, scored on the next 16 months)", color=ink_2)
    ax.set_xticks(months[12::3][:8], [m.strftime("%Y-%m") for m in months[12::3][:8]])
    ax.tick_params(colors=muted, length=0)
    ax.grid(color=grid, lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(axis)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout()
    fig.savefig(OUT / "level_backtest.png", dpi=200)


def main(argv=()):
    OUT.mkdir(parents=True, exist_ok=True)
    if "--rolling-premiums" in argv:
        rolling_premiums()
    else:
        holdout_tables()
        if "--holdout-only" not in argv:
            rolling()
            figure()
    print("DONE")

## 5. Geographic transfer experiments

Whole-subzone and whole-MRT-catchment stress tests for the GNNs. Every fitting and tuning rent in entire areas is withheld while their building attributes stay visible, and the models are scored on those areas' January–June 2024 rentals. This is a different setting from the local-block forecasting in the rest of the project, so compare these numbers only with each other.

`3_evaluation.ipynb` shows the results (end of the notebook) and `2_models.ipynb` has the optional reproduction cell (end of Part 2.4, off by default). Graph inputs are in `data/transfer/`; frozen results in `outputs/gnn_transfer/`; new runs go to `outputs/gnn_transfer/runs/` (git-ignored). `transfer.main(["--experiment", "both", "--smoke", "--seeds", "0"])` is a quick execution check; drop `--smoke` for the full experiments (a few minutes each on a laptop CPU).

### Setup

| | Hierarchy | MRT catchments |
|---|---|---|
| Hidden labels | 31 entire subzones | 18 entire nearest-station catchments |
| Selection | Seed 17; observed before July 2023; keep an observed sister subzone in each planning area | Seed 417; observed before July 2023; keep an observed adjacent catchment |
| Inner fit / tuning | Before July 2023 / July–December 2023 | Same |
| Outer fit / evaluation | Before January 2024 / January–June 2024 | Same |
| Seeds | 0, 1 | 0, 1 |
| Width / dropout | 32 / 0.1 | 32 / 0.1 |
| Maximum epochs / patience | 512 / 35 | 640 / 35 |
| Optimizer | AdamW, learning rate 0.001, weight decay 0.001 | Same |
| Market level / loss | Fold-local damped forecast / dollar-weighted squared error | Same |

Withheld rents are excluded from fitting, preprocessing, the market level and tuning. Known building nodes stay visible, and the models pass building attributes, not rents, through their pooled representations. A block's station is the nearest by straight-line distance, with no radius cutoff (not a walking-time catchment). The station network is frozen at January 2021.

The hierarchy compares a matched MLP with subzone → planning-area pooling. MRT compares the local MLP, catchment pooling without rail, one and three rail steps, shuffled rail connections, and geographic connections (each station linked to its two nearest, symmetrized: 151 edges versus 137 rail edges).

### Results

Held-area RMSE, predictions averaged over the two seeds before scoring:

| Model | RMSE |
|---|---|
| *31 subzones, 3,836 rentals* | |
| Matched MLP (`MLP_geo`) | 532.3 |
| Hierarchical pooling (`Hierarchy_static`) | **525.4** |
| *18 MRT catchments, 4,429 rentals* | |
| Matched MLP (`MLP_station`) | 509.8 |
| Catchment pooling without rail (`Catchment_only`) | 509.9 |
| One rail step (`Rail_1`) | 508.0 |
| Geographic connections (`Geographic_3`) | 506.6 |
| Three rail steps (`Rail_3`) | **505.5** |
| Shuffled rail connections (`Shuffled_3`) | 515.0 |

- The hierarchy beats its MLP by 6.8, but the planning-area bootstrap range (−23.1 to +1.2) includes zero.
- Three rail steps beat catchment pooling alone (−4.4, range −7.6 to −1.3), and shuffling the rail connections makes things worse. Geographic connections recover most of the gain, so this does not show that rail topology itself matters.
- Against the matched MLP, three rail steps are not clearly better (−4.3, range −9.9 to +2.0).
- A fresh full rerun (PyTorch 2.15, 2026-11) gave 525.5 / 532.7 for the hierarchy and kept `Rail_3` best on MRT (506.0). Its lead over `MLP_station` shrank to 1.1, so treat the MRT margins as run-dependent.

These are exploratory backtests: one withheld-area assignment per experiment, on windows that were inspected during development. The bootstrap holds predictions fixed, so it does not include retraining or new area selections. The results suggest broader geographic connections may help where an area has no rental history. They do not show a reliable competition-score gain.

### Results files (`outputs/gnn_transfer/`)

- `table.csv`: held-area scores for all eight models.
- `ensemble_metrics.csv`: all-row and held-area scores.
- `seed_metrics.csv`, `selected_epochs.csv`, `held_groups.csv`: individual runs, chosen epoch counts and withheld groups.
- `bootstrap.csv`: cluster bootstrap ranges; negative differences favour the graph model. Hierarchy clusters are planning areas, MRT clusters are station catchments.
- `cluster_errors.csv`: per-group error totals that reconstruct the held-area RMSE, MAE and bias and support the bootstrap.
- `context_metrics.csv`: ordinary-forecasting and random-block results for the same models, where neither approach consistently helps.
- `protocols.json`, `extraction_checks.json`, `smoke_checks.json`: original settings and the checks that this code reproduces the original experiments.

The models were extracted from the original exploratory scripts, which are not part of this repo (`outputs/gnn_transfer/provenance.json` lists their hashes). The checks confirmed identical held-row selections, preprocessing, parameter initialization and forward outputs, with gradients matching within float32 tolerance.

In [ ]:
%%module transfer_models
"""Original transfer architectures with graph data passed explicitly.

No experiment runs at import time. Common initialization and layer order are
preserved so the extracted models can be checked against the original runs.
"""
import torch
from torch import nn

HIDDEN = 32

def group_mean(x,index,size):
    total=x.new_zeros((size,x.shape[1])).index_add_(0,index,x)
    count=torch.bincount(index,minlength=size).to(x.dtype).clamp_min(1).unsqueeze(1)
    return total/count


class HierarchicalRentalModel(nn.Module):
    """Same transforms in each arm; pooling and trainable area IDs are toggled."""
    def __init__(self,kind,bundle,graph):
        super().__init__()
        node_dim = bundle["nodes"].shape[1]
        record_dim = bundle["parts"][0]["record"].shape[1]
        BLOCK_SUB, SUB_AREA = graph["block_sub"], graph["sub_area"]
        BLOCK_AREA = SUB_AREA[BLOCK_SUB]
        NS, NA = graph["n_subzones"], graph["n_areas"]
        self.n_subzones, self.n_areas = NS, NA
        self.kind=kind
        self.pool=kind.startswith('Hierarchy')
        self.learn_area=kind.endswith('area_embeddings')
        self.projection=nn.Linear(node_dim,HIDDEN)
        self.sub_up=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN) for _ in range(2)])
        self.area_up=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN) for _ in range(2)])
        self.sub_self=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN) for _ in range(2)])
        self.area_down=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN,bias=False) for _ in range(2)])
        self.sub_down=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN,bias=False) for _ in range(2)])
        self.block_self=nn.ModuleList([nn.Linear(HIDDEN,HIDDEN) for _ in range(2)])
        self.gates=nn.ModuleList([nn.Linear(2*HIDDEN,HIDDEN) for _ in range(2)])
        for gate in self.gates:nn.init.constant_(gate.bias,-2.)
        self.head=nn.Sequential(nn.Linear(HIDDEN+record_dim,HIDDEN),nn.ReLU(),nn.Dropout(.1),nn.Linear(HIDDEN,1))
        self.drop=nn.Dropout(.1)
        # Zero initialization and no random embedding initializer keep common
        # parameter initialization and dropout streams aligned between arms.
        self.sub_embedding=nn.Parameter(torch.zeros(NS,HIDDEN),requires_grad=self.learn_area)
        self.area_embedding=nn.Parameter(torch.zeros(NA,HIDDEN),requires_grad=self.learn_area)
        self.register_buffer('block_sub',BLOCK_SUB)
        self.register_buffer('sub_area',SUB_AREA)
        self.register_buffer('block_area',BLOCK_AREA)

    def forward(self,bundle,part):
        nodes = bundle["nodes"]
        z=self.drop(torch.relu(self.projection(nodes)))
        for depth in range(2):
            if self.pool:
                # Explicit bottom-up pass: blocks -> subzones -> planning areas.
                sub=group_mean(z,self.block_sub,self.n_subzones)
                sub=torch.relu(self.sub_up[depth](sub)+self.sub_embedding)
                area=group_mean(sub,self.sub_area,self.n_areas)
                area=torch.relu(self.area_up[depth](area)+self.area_embedding)
                # Top-down pass completes the hierarchy within this round.
                sub=torch.relu(self.sub_self[depth](sub)+self.area_down[depth](area[self.sub_area]))
                context=self.sub_down[depth](sub[self.block_sub])
            else:
                # Each block has its own virtual group states. The exact same
                # transforms/area ID embeddings are active, with no pooling
                # or message transfer from other blocks.
                sub=torch.relu(self.sub_up[depth](z)+self.sub_embedding[self.block_sub])
                area=torch.relu(self.area_up[depth](sub)+self.area_embedding[self.block_area])
                sub=torch.relu(self.sub_self[depth](sub)+self.area_down[depth](area))
                context=self.sub_down[depth](sub)
            gate=torch.sigmoid(self.gates[depth](torch.cat([z,context],dim=1)))
            z=self.drop(torch.relu(self.block_self[depth](z)+gate*context))
        return self.head(torch.cat([z[part['ids']],part['record']],dim=1)).squeeze(-1)


class StationModel(nn.Module):
    """Pool first, propagate on the small graph, then read back to each block.

    All arms have identical active parameters and initialization. The local MLP
    applies the same transforms to a private virtual station for each block.
    """
    def __init__(self, kind, bundle, graph, hidden=32):
        super().__init__()
        self.graph = graph
        self.kind = kind
        self.projection = nn.Linear(bundle['nodes'].shape[1], hidden)
        self.station_projection = nn.Linear(bundle['station'].shape[1], hidden)
        self.up = nn.Linear(hidden, hidden)
        self.roots = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(3)])
        self.messages = nn.ModuleList([nn.Linear(hidden, hidden, bias=False) for _ in range(3)])
        self.station_gates = nn.ModuleList([nn.Linear(2*hidden, hidden) for _ in range(3)])
        self.down = nn.Linear(hidden, hidden, bias=False)
        self.local = nn.Linear(hidden, hidden)
        self.gate = nn.Linear(2*hidden, hidden)
        self.head = nn.Sequential(nn.Linear(hidden+bundle['parts'][0]['record'].shape[1], hidden),
                                  nn.ReLU(), nn.Dropout(.1), nn.Linear(hidden, 1))
        self.drop = nn.Dropout(.1)
        for gate in [*self.station_gates, self.gate]:
            nn.init.constant_(gate.bias, -2.)

    def forward(self, bundle, part):
        z = self.drop(torch.relu(self.projection(bundle['nodes'])))
        local = self.kind == 'MLP_station'
        if local:
            pooled = z
            descriptors = bundle['station'][self.graph["membership"]]
        else:
            pooled = z.new_zeros((self.graph["n_stations"], z.shape[1])).index_add_(0, self.graph["membership"], z)/self.graph["counts"]
            descriptors = bundle['station']
        station = torch.relu(self.up(pooled) + self.station_projection(descriptors))
        for layer, (root, message, gate) in enumerate(zip(self.roots, self.messages, self.station_gates)):
            propagate = self.kind in self.graph["adjacency"] and (self.kind != 'Rail_1' or layer == 0)
            incoming = message(self.graph["adjacency"][self.kind] @ station if propagate else station)
            amount = torch.sigmoid(gate(torch.cat([station, incoming], 1)))
            station = torch.relu(root(station)+amount*incoming)
        context = self.down(station if local else station[self.graph["membership"]])
        amount = torch.sigmoid(self.gate(torch.cat([z, context], 1)))
        z = self.drop(torch.relu(self.local(z)+amount*context))
        return self.head(torch.cat([z[part['ids']], part['record']], 1)).squeeze(-1)

In [ ]:
%%module transfer
"""Run the report's whole-subzone or whole-MRT-catchment experiments.

Example: main(["--experiment", "hierarchy", "--smoke"])
Reads data/train.csv and the graph inputs in data/transfer/. Writes outputs/gnn_transfer/runs/,
never the frozen results in outputs/gnn_transfer/.
"""
import argparse
import json
import random
import sys
from pathlib import Path

import torch  # Import before libraries that may load another OpenMP runtime.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from hdb_rentals.transfer_models import HierarchicalRentalModel, StationModel

INPUTS = ROOT / 'data' / 'transfer'
KEY = ['TOWN', 'BLOCK', 'STREET']
MODELS = {'hierarchy': ['MLP_geo', 'Hierarchy_static'],
          'mrt': ['MLP_station', 'Catchment_only', 'Rail_1', 'Rail_3',
                  'Shuffled_3', 'Geographic_3']}


def load_data(root):
    blocks = pd.read_csv(INPUTS / 'block_nodes.csv.gz').sort_values('block_id').reset_index(drop=True)
    assert np.array_equal(blocks.block_id, np.arange(len(blocks)))
    for c in ['MAX_FLOOR', 'YEAR_COMPLETED']:
        blocks[c] = pd.to_numeric(blocks[c], errors='coerce').replace(-1, np.nan)
    rent = pd.read_csv(root / 'data/train.csv')
    for c in KEY:
        rent[c] = rent[c].astype('string').str.strip().str.lower()
        blocks[c] = blocks[c].astype('string').str.strip().str.lower()
    rent['row_id'] = np.arange(len(rent))
    rent['RENT_APPROVAL_DATE'] = pd.to_datetime(rent.RENT_APPROVAL_DATE, format='%Y-%m')
    rent['MONTH_INDEX'] = (rent.RENT_APPROVAL_DATE.dt.year-2021)*12 + rent.RENT_APPROVAL_DATE.dt.month-1
    rent['FLAT_AGE'] = rent.RENT_APPROVAL_DATE.dt.year-rent.LEASE_COMMENCE_DATE
    rent['FLAT_TYPE'] = rent.FLAT_TYPE.str.lower().str.strip().str.replace(r'[\s-]+', '-', regex=True)
    rent['FLAT_MODEL'] = rent.FLAT_MODEL.str.lower().str.strip()
    rent = rent.merge(blocks[KEY+['block_id']], on=KEY, how='left', validate='m:1', sort=False)
    assert rent.block_id.notna().all() and rent.row_id.tolist() == list(range(len(rent)))
    return blocks, rent


def build_graph(blocks, experiment):
    if experiment == 'hierarchy':
        subzones, areas = sorted(blocks.SUBZONE.unique()), sorted(blocks.PLANNING_AREA.unique())
        assert blocks.groupby('SUBZONE').PLANNING_AREA.nunique().eq(1).all()
        mapping = blocks[['SUBZONE', 'PLANNING_AREA']].drop_duplicates().set_index('SUBZONE').loc[subzones]
        return dict(block_sub=torch.tensor(pd.Categorical(blocks.SUBZONE, categories=subzones).codes.astype('int64')),
                    sub_area=torch.tensor(pd.Categorical(mapping.PLANNING_AREA, categories=areas).codes.astype('int64')),
                    n_subzones=len(subzones), n_areas=len(areas))
    stations = pd.read_csv(INPUTS / 'mrt_nodes.csv').sort_values('amenity_id').reset_index(drop=True)
    edges = pd.read_csv(INPUTS / 'mrt_rail_edges.csv')
    xy = np.radians(stations[['LATITUDE', 'LONGITUDE']].to_numpy())
    distance, ids = BallTree(xy, metric='haversine').query(
        np.radians(blocks[['LATITUDE', 'LONGITUDE']]), k=1)
    ids, distance = ids[:, 0], distance[:, 0]*6371.0088
    n = len(stations)
    rail = np.zeros((n, n), dtype='float32')
    rail[edges.source, edges.target] = rail[edges.target, edges.source] = 1
    _, neighbours = BallTree(xy, metric='haversine').query(xy, k=3)
    geographic = np.zeros_like(rail)
    geographic[np.arange(n)[:, None], neighbours[:, 1:]] = 1
    geographic = np.maximum(geographic, geographic.T)
    perm = np.random.default_rng(731).permutation(n)
    def normalize(a):
        return torch.tensor(a / np.maximum(a.sum(1, keepdims=True), 1))
    membership = torch.tensor(ids, dtype=torch.long)
    return dict(stations=stations, ids=ids, distance=distance, rail=rail,
                membership=membership, n_stations=n,
                counts=torch.bincount(membership, minlength=n).float().clamp_min(1).unsqueeze(1),
                adjacency={'Rail_1': normalize(rail), 'Rail_3': normalize(rail),
                           'Shuffled_3': normalize(rail[perm][:, perm]),
                           'Geographic_3': normalize(geographic)})


def split_data(blocks, rent, graph, experiment):
    """Choose held groups from pre-July-2023 activity, never future rents."""
    date = rent.RENT_APPROVAL_DATE
    observed = blocks[blocks.block_id.isin(rent.loc[date < '2023-07-01', 'block_id'])]
    if experiment == 'hierarchy':
        mapping = observed[['SUBZONE', 'PLANNING_AREA']].drop_duplicates().sort_values('SUBZONE')
        remaining = mapping.groupby('PLANNING_AREA').SUBZONE.nunique().to_dict()
        held = []
        for row in mapping.iloc[np.random.default_rng(17).permutation(len(mapping))].itertuples(index=False):
            if remaining[row.PLANNING_AREA] > 1:
                held.append(row.SUBZONE)
                remaining[row.PLANNING_AREA] -= 1
            if len(held) == int(np.floor(.2*len(mapping))):
                break
        held_blocks = blocks.loc[blocks.SUBZONE.isin(held), 'block_id'].to_numpy()
    else:
        observed_stations = np.unique(graph['ids'][observed.block_id])
        candidates = [int(i) for i in observed_stations if np.any(graph['rail'][i, observed_stations])]
        held = []
        for station in np.random.default_rng(417).permutation(candidates):
            proposed = set(held+[int(station)])
            remaining = np.array([i for i in observed_stations if i not in proposed])
            if all(np.any(graph['rail'][i, remaining]) for i in proposed):
                held.append(int(station))
            if len(held) == int(np.floor(.2*len(candidates))):
                break
        held_blocks = np.flatnonzero(np.isin(graph['ids'], held))
    retained = ~rent.block_id.isin(held_blocks)
    frames = [rent[(date < '2023-07-01') & retained].copy(),
              rent[(date >= '2023-07-01') & (date < '2024-01-01') & retained].copy(),
              rent[(date < '2024-01-01') & retained].copy(),
              rent[(date >= '2024-01-01') & (date < '2024-07-01')].copy()]
    for f in frames[:3]:
        assert not f.block_id.isin(held_blocks).any()
    assert frames[0].RENT_APPROVAL_DATE.max() < frames[1].RENT_APPROVAL_DATE.min()
    assert frames[2].RENT_APPROVAL_DATE.max() < frames[3].RENT_APPROVAL_DATE.min()
    frames[3]['held_out'] = frames[3].block_id.isin(held_blocks)
    return frames, held


def preprocessor(numeric, categorical):
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                          ('scale', StandardScaler())]), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
    ], sparse_threshold=0)


def prepare(blocks, fit, val, graph, experiment):
    from hdb_rentals.rent_adjustment import MarketRentAdjustment
    spec = json.loads((INPUTS / 'feature_spec.json').read_text())
    bp = preprocessor(spec['numeric'], spec['categorical'])
    fit_blocks = np.sort(fit.block_id.unique())
    bp.fit(blocks.iloc[fit_blocks])
    nodes, extra = bp.transform(blocks), {}
    if experiment == 'mrt':
        stations, ids = graph['stations'], graph['ids']
        columns = ['LATITUDE', 'LONGITUDE', 'n_lines', 'is_interchange'] + [c for c in stations if c.startswith('line_')]
        sx = stations[columns].to_numpy(float)
        visible = np.unique(ids[fit.block_id.unique()])
        sx = (sx-sx[visible].mean(0))/np.maximum(sx[visible].std(0), 1e-6)
        access = np.column_stack([sx[ids], graph['distance']])
        # Preserve original row order in these means for numerical parity.
        access_fit = access[fit.block_id.unique()]
        access = (access-access_fit.mean(0))/np.maximum(access_fit.std(0), 1e-6)
        nodes = np.column_stack([nodes, access])
        extra['station'] = torch.tensor(sx.astype('float32'))
    rp = preprocessor(['FLOOR_AREA_SQM', 'FLAT_AGE'], ['FLAT_TYPE', 'FLAT_MODEL'])
    rp.fit(fit)
    adjustment = MarketRentAdjustment('damped').fit(fit)
    scale = adjustment.multipliers(fit.MONTH_INDEX)
    adjusted = fit.MONTHLY_RENT.to_numpy()/scale
    mean, std = float(adjusted.mean()), float(adjusted.std(ddof=1))
    def pack(frame):
        s, y = adjustment.multipliers(frame.MONTH_INDEX), frame.MONTHLY_RENT.to_numpy(float)
        return dict(ids=torch.tensor(frame.block_id.to_numpy(int)),
                    record=torch.tensor(rp.transform(frame).astype('float32')),
                    y=y, scale=s, scaled_y=torch.tensor(((y/s-mean)/std).astype('float32')),
                    weight=torch.tensor((s*s/np.mean(scale*scale)).astype('float32')))
    return dict(nodes=torch.tensor(nodes.astype('float32')), parts=[pack(fit), pack(val)],
                mean=mean, std=std, **extra)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@torch.no_grad()
def predict(model, bundle):
    model.eval()
    val = bundle['parts'][1]
    return np.maximum(model(bundle, val).numpy()*bundle['std']+bundle['mean'], 0)*val['scale']


def fit_model(kind, bundle, graph, experiment, seed, epochs, tune=False):
    seed_all(seed)
    factory = HierarchicalRentalModel if experiment == 'hierarchy' else StationModel
    model = factory(kind, bundle, graph)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=.001, weight_decay=.001)
    fit, best, best_epoch, stale, history = bundle['parts'][0], float('inf'), epochs, 0, []
    for epoch in range(1, epochs+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = ((model(bundle, fit)-fit['scaled_y']).square()*fit['weight']).mean()
        if experiment == 'hierarchy':
            loss = loss+.1*(model.sub_embedding.square().mean()+model.area_embedding.square().mean())
        assert torch.isfinite(loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
        optimizer.step()
        row = dict(epoch=epoch, loss=float(loss.detach()))
        if tune:
            score = float(np.sqrt(np.mean((predict(model, bundle)-bundle['parts'][1]['y'])**2)))
            row['inner_rmse'] = score
            if score < best-.01:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        history.append(row)
        if tune and stale >= 35:
            break
    return model, best_epoch, history


def score_predictions(predictions):
    rows = []
    for model, f in predictions.groupby('model'):
        for label, g in [('all', f), ('held_out', f[f.held_out])]:
            error = g.prediction-g.MONTHLY_RENT
            rows.append(dict(model=model, slice=label, n_rows=len(g),
                             rmse=np.sqrt(np.mean(error**2)), mae=error.abs().mean(), bias=error.mean()))
    return pd.DataFrame(rows)


def cluster_errors(ensemble, blocks, graph, experiment):
    """Sufficient statistics for the held-area scores and fixed-prediction bootstrap."""
    f = ensemble[ensemble.held_out].copy()
    groups = blocks.PLANNING_AREA.to_numpy() if experiment == 'hierarchy' else graph['ids']
    f['cluster'] = groups[f.block_id]
    error = f.prediction-f.MONTHLY_RENT
    f['n'], f['sse'], f['sae'], f['error_sum'] = 1, error**2, error.abs(), error
    return f.groupby(['model', 'cluster'], as_index=False)[['n', 'sse', 'sae', 'error_sum']].sum()


def bootstrap(stats, experiment):
    """Resample groups, keeping trained predictions fixed; negative delta improves."""
    model = 'Hierarchy_static' if experiment == 'hierarchy' else 'Rail_3'
    references = ['MLP_geo'] if experiment == 'hierarchy' else [
        'MLP_station', 'Catchment_only', 'Shuffled_3', 'Geographic_3']
    rng, rows = np.random.default_rng(981), []
    for reference in references:
        if not {model, reference} <= set(stats.model):
            continue
        a = stats[stats.model == model].set_index('cluster')
        b = stats[stats.model == reference].set_index('cluster').loc[a.index]
        np.testing.assert_array_equal(a.n, b.n)
        values = np.column_stack([a.n, a.sse, b.sse])
        draws = values[rng.integers(0, len(a), size=(2000, len(a)))].sum(1)
        delta = np.sqrt(draws[:, 1]/draws[:, 0])-np.sqrt(draws[:, 2]/draws[:, 0])
        rows.append(dict(model=model, reference=reference, clusters=len(a),
                         delta=np.sqrt(a.sse.sum()/a.n.sum())-np.sqrt(b.sse.sum()/b.n.sum()),
                         lower=np.quantile(delta, .025), upper=np.quantile(delta, .975)))
    return pd.DataFrame(rows)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--experiment', choices=['hierarchy', 'mrt', 'both'], default='both')
    ap.add_argument('--root', type=Path, default=ROOT)
    ap.add_argument('--output', type=Path, default=ROOT / 'outputs' / 'gnn_transfer' / 'runs')
    ap.add_argument('--seeds', default='0,1')
    ap.add_argument('--models', help='Optional comma-separated subset of model names')
    ap.add_argument('--smoke', action='store_true', help='700 rows per split, three epochs; execution check only')
    args = ap.parse_args(argv)
    blocks, rent = load_data(args.root.resolve())
    experiments = ['hierarchy', 'mrt'] if args.experiment == 'both' else [args.experiment]
    for experiment in experiments:
        threads, epochs = (2, 512) if experiment == 'hierarchy' else (4, 640)
        torch.set_num_threads(threads)
        graph = build_graph(blocks, experiment)
        frames, held = split_data(blocks, rent, graph, experiment)
        out = args.output / ('smoke' if args.smoke else 'full') / experiment
        out.mkdir(parents=True, exist_ok=True)
        membership = rent[['row_id', 'block_id', 'RENT_APPROVAL_DATE']].copy()
        for name, f in zip(['inner_fit', 'inner_tune', 'outer_fit', 'outer_compare'], frames):
            membership[name] = membership.row_id.isin(f.row_id)
        membership.to_csv(out / 'membership.csv', index=False)
        kinds = args.models.split(',') if args.models else MODELS[experiment]
        assert set(kinds) <= set(MODELS[experiment])
        config = dict(experiment=experiment, smoke=args.smoke, seeds=args.seeds, models=kinds,
                      held_groups=held, threads=threads, max_epochs=3 if args.smoke else epochs,
                      market='fold-local damped', patience=35, hidden=32,
                      held_labels_in_fit_or_tune=0)
        (out / 'config.json').write_text(json.dumps(config, indent=2))
        if args.smoke:
            frames = [f.sample(min(len(f), 700), random_state=0).sort_index() for f in frames]
        inner = prepare(blocks, *frames[:2], graph, experiment)
        outer = prepare(blocks, *frames[2:], graph, experiment)
        predictions, selections = [], []
        for kind in kinds:
            for seed in map(int, args.seeds.split(',')):
                _, count, history = fit_model(kind, inner, graph, experiment, seed, config['max_epochs'], tune=True)
                fitted, _, _ = fit_model(kind, outer, graph, experiment, seed, count)
                f = frames[3][['row_id', 'block_id', 'MONTHLY_RENT', 'held_out']].copy()
                f['prediction'], f['model'], f['seed'] = predict(fitted, outer), kind, seed
                predictions.append(f)
                selections.append(dict(model=kind, seed=seed, epochs=count, tuning_epochs=len(history),
                                       parameters=sum(p.numel() for p in fitted.parameters() if p.requires_grad)))
                print(experiment, kind, seed, 'epochs', count, flush=True)
        predictions = pd.concat(predictions, ignore_index=True)
        assert not predictions.duplicated(['model', 'seed', 'row_id']).any()
        predictions.to_csv(out / 'predictions.csv.gz', index=False)
        pd.DataFrame(selections).to_csv(out / 'selected_epochs.csv', index=False)
        ensemble = predictions.groupby(['model', 'row_id', 'block_id', 'held_out'], as_index=False).agg(
            MONTHLY_RENT=('MONTHLY_RENT', 'first'), prediction=('prediction', 'mean'))
        scores = score_predictions(ensemble)
        scores.to_csv(out / 'ensemble_metrics.csv', index=False)
        stats = cluster_errors(ensemble, blocks, graph, experiment)
        stats.to_csv(out / 'cluster_errors.csv', index=False)
        bootstrap(stats, experiment).to_csv(out / 'bootstrap.csv', index=False)
        print(scores[scores['slice'] == 'held_out'].to_string(index=False), flush=True)